# K2b — 최신 LLM 실제 응답으로 F1 base 재학습 · 새 조합 판정

## 왜 하나

배포 판정기(I3 조합: F1 base + I2)는 최신 Claude 응답의 환각을 10~14%밖에 잡지 못했고(M1·K1), 합성 모순 학습(K1)으로도 나아지지 않았다.
K2a에서 **최신 LLM이 실제로 쓴 응답**(학습용, Sonnet 문장 정답)을 모았다. 이것을 학습 데이터에 더해 경량 판정기가 최신 LLM 환각을 배우는지 본다.
임계값도 RAGTruth 기준(A)과 **최신 LLM 멀쩡한 응답 기준(B, cal-M 10%)** 두 가지로 본다.

## 무엇을 하나

1. F1 노트북과 **같은 데이터·설정·시드 42**로 DeBERTa-v3-base 학습, 학습 지점에 K2a 학습 응답의 모든 판정 지점을 더함(정답 문구 위치부터 환각, R4b와 같은 매핑).
   **구현 보충(실행 전)**: 학습 시간을 F1의 약 2배로 묶기 위해 최신 LLM 지점 수가 기존 학습 지점 수를 넘으면 응답 단위로 무작위(시드 2032)로 줄인다. 후보 1개, 선택 없음
2. 새 F1 점수 + 저장된 I2 점수 → I3와 같은 방식(cal-L 표준화) → **새 조합**. 임계값 A = cal-L 서비스 10%, 임계값 B = cal-M 멀쩡한 응답 표시율 10%(등각)
3. RAGTruth cal-L에서 기존 조합과 비교(K2-3), dev-L은 보고만
4. 최신 LLM 세트(dev-M·cal-M·m1_fresh·m1)를 게이트웨이 `judge.py`로 새 F1만 다시 판정(저장 지점·I2 점수 그대로) → K2-0·K2-1·K2-2

## 사전 정의 기준 (SSOT §6 K2, 실행 전 고정)

- **K2-0(유효성)**: train 정답 파싱 ≥ 90%, train H 문장 ≥ 500, cal-M 멀쩡한 응답 ≥ 400, dev-M 환각 응답 ≥ 150 — 미달이면 보류
- **K2-1(주)**: dev-M 환각 응답 탐지율(아무 지점) **새 조합(B) − 기존 조합(B) ≥ +0.10**, 응답 쌍체 부트스트랩 95% 하한 > 0
- **K2-2(안전, 최신 LLM)**: dev-M 멀쩡한 응답 표시율 새 조합(B) ≤ 15%
- **K2-3(안전, RAGTruth)**: 임계값 A에서 cal-L 8토큰 서비스 10% 원래 정답 탐지율 ≥ 기존 조합 − 0.02(0.543 → 0.523 이상), C1도 ≥ −0.02
- **채택 = K2-0~3 모두** → 배포 판정기 = 새 조합
- **임계값 B 채택(모델 채택과 별개)**: 배포 판정기(채택이면 새, 아니면 기존 조합)의 dev-M 멀쩡한 응답 표시율이 B에서 ≤ 15%면 서비스 임계값을 B로

> **런타임: A100**(L4 가능, 20GB 미만이면 첫 셀에서 멈춤). 학습 약 2.5~3.5시간 + 최신 LLM 세트 다시 판정 약 1시간. 끊기면 다시 모두 실행 — 끝난 단계는 드라이브 `k2/`에서 건너뛴다(학습 도중 끊기면 학습은 처음부터).
> 먼저: **K2a**(`k2/`의 train·calM·devM 파일)와 `m1_fresh/`. **실행 전에 드라이브 휴지통 비우기**(가중치 약 370MB).
> 결과: 드라이브 `k2/`(가중치 `k2/judge_base` — 배포 `final/`은 그대로)와 `k2_results.zip`.


## 1. 환경 준비

In [ ]:
!pip -q install pysbd sentence-transformers tiktoken
!git clone --depth 1 https://github.com/ParticleMedia/RAGTruth.git 2>/dev/null || echo 'RAGTruth already cloned'

import json, time, random, os, gc, copy
import numpy as np, pandas as pd, torch, pysbd, tiktoken
from sklearn.metrics import roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/Colab Notebooks/HalluGate' if USE_DRIVE else '.'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
STUDENT_DIR = os.path.join(OUT_DIR, 'student_vark')
R0_DIR = os.path.join(OUT_DIR, 'r0')
R1_DIR = FINAL_DIR = K2_DIR = os.path.join(OUT_DIR, 'k2')          # 배포 가중치(final/)와 따로
OLD_DIR, I2_DIR = os.path.join(OUT_DIR, 'final'), os.path.join(OUT_DIR, 'i2')
# 최신 LLM 세트: (태그, 폴더, 파일 접미사)
MSETS = [('devM', K2_DIR, 'devM'), ('calM', K2_DIR, 'calM'), ('fresh', os.path.join(OUT_DIR, 'm1_fresh'), 'devL'), ('m1', os.path.join(OUT_DIR, 'm1'), 'devL')]
for p in [os.path.join(OLD_DIR, 'scores_base.npy'), os.path.join(I2_DIR, 'scores_i2.npy'), os.path.join(OLD_DIR, 'judge_base', 'judge_config.json')] + \
         [os.path.join(K2_DIR, f'{n}_train.json') for n in ['cases', 'gen', 'label']] + \
         [os.path.join(d, f'{n}_{s}.json') for _, d, s in MSETS for n in ['cases', 'gen', 'label', 'scores']]:
    assert os.path.exists(p), f'{p} 없음 (K2a·m1_fresh 먼저)'
R4C_DIR = os.path.join(OUT_DIR, 'r4c')
R1B_DIR, R1C_DIR, R4_DIR, R4B_DIR = (os.path.join(OUT_DIR, d) for d in ['r1b', 'r1c', 'r4', 'r4b'])
os.makedirs(R1_DIR, exist_ok=True)
for p in [os.path.join(R4B_DIR, 'labels_train.json'), os.path.join(R4_DIR, 'r4_corrected_labels.json')]:
    assert os.path.exists(p), f'{p} 없음'
assert os.path.isdir(STUDENT_DIR), f'{STUDENT_DIR} 없음 (MiniLM 학습 시작점)'
# 드라이브 쓰기 점검: R4c에서는 용량 부족으로 가중치가 조용히 저장되지 않았다
import shutil
_t = os.path.join(FINAL_DIR, '_write_test.bin')
try:
    with open(_t, 'wb') as f: f.write(os.urandom(50 * 2**20))
    ok = os.path.getsize(_t) == 50 * 2**20; os.remove(_t)
except OSError as e:
    ok = False; print('드라이브 쓰기 실패:', e)
print('드라이브 50MB 쓰기', '정상' if ok else '실패 — 휴지통 비우기·용량 확인. 가중치는 마지막 셀의 zip 다운로드로 받는다')
print('device', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
assert DEVICE == 'cuda' and torch.cuda.get_device_properties(0).total_memory >= 20 * 2**30, \
    f"GPU 메모리 부족: {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'} — 런타임 유형을 A100으로 바꾸고 다시 실행"

## 2. 데이터와 보정 학습 정답 (F1과 같음)

In [ ]:
BASE = 'RAGTruth/dataset'
responses = [json.loads(l) for l in open(f'{BASE}/response.jsonl')]
sources   = {json.loads(l)['source_id']: json.loads(l) for l in open(f'{BASE}/source_info.jsonl')}
for r in responses:
    r['task_type'] = sources[r['source_id']]['task_type']

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

TASKS = ['QA', 'Summary', 'Data2txt']
train_pool = [r for r in responses if r['split'] == 'train']          # test 분할은 사용하지 않음
random.seed(SEED)
dev = []
for tt in TASKS:
    dev += random.sample([r for r in train_pool if r['task_type'] == tt], 120)
dev_ids = {r['id'] for r in dev}
rest = [r for r in train_pool if r['id'] not in dev_ids]
rng = random.Random(11)
teacher_resp = []
for tt in TASKS:
    teacher_resp += rng.sample([r for r in rest if r['task_type'] == tt], 200)
rng.shuffle(teacher_resp)
val = teacher_resp[:int(len(teacher_resp) * 0.15)]                   # student 학습에 쓰지 않은 검증 분할
assert not (dev_ids & {r['id'] for r in val})
print(f'dev {len(dev)} 응답 | 검증 {len(val)} 응답')

In [ ]:
# 학습 응답: S2/S2-b의 gold-large와 같은 구성 (teacher 학습 510 + 추가 2,490). dev·검증과 겹치지 않음을 확인한다.
teacher_train = teacher_resp[len(val):]
used = dev_ids | {r['id'] for r in teacher_resp}
gl_extra = rng.sample([r for r in rest if r['id'] not in used], 3000 - len(teacher_train))
train = teacher_train + gl_extra
tr_ids = {r['id'] for r in train}
assert not (tr_ids & dev_ids) and not (tr_ids & {r['id'] for r in val}), '학습과 dev/검증 겹침'
print(f'학습 {len(train)} 응답 (dev·검증과 겹침 없음)')

In [ ]:
ENC = tiktoken.get_encoding('cl100k_base')
seg = pysbd.Segmenter(language='en', clean=False, char_span=True)

def tokenize_offsets(text):
    ids = ENC.encode(text)
    dec, starts = ENC.decode_with_offsets(ids)
    if dec != text:                       # 드문 경우: 멀티바이트 문자 경계 — 누적 디코드로 대체
        starts, pos = [], 0
        for i in range(len(ids)):
            starts.append(pos); pos = len(ENC.decode(ids[:i + 1]))
    ends = starts[1:] + [len(text)]
    return starts, ends

def prep(r):
    text = r['response']
    ts, te = tokenize_offsets(text)
    # 토큰의 '내용 시작' 위치: cl100k 토큰은 앞 공백을 포함하므로(" The"), 공백을 건너뛴 위치로 문장·환각 구간에 배정한다.
    # (v1에서는 공백 위치로 배정해 문장 경계 토큰이 앞 문장에 붙는 오류가 발생 — 발생 위치의 18%)
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) for a, b in zip(ts, te)]
    cs = [c if c < b else a for c, a, b in zip(cs, ts, te)]          # 공백뿐인 토큰
    spans = [(l['start'], l['end']) for l in (r.get('labels') or [])]
    bad = [j for j in range(len(ts)) if any(min(te[j], b) > max(cs[j], a) for a, b in spans)]
    sents = []
    for sp in seg.segment(text):
        if not sp.sent.strip(): continue
        a, b = sp.start, sp.end
        tok = [j for j in range(len(ts)) if a <= cs[j] < b]
        if not tok: continue
        y = int(any(min(b, e) > max(a, s) for s, e in spans))
        sents.append({'a': a, 'b': b, 't0': tok[0], 't1': tok[-1], 'y': y})
    sent_of = {t: sn['t0'] for sn in sents for t in range(sn['t0'], sn['t1'] + 1)}
    return {'id': str(r['id']), 'task_type': r['task_type'], 'gen_model': r['model'],
            'ref_key': r['source_id'], 'text': text, 'ts': ts, 'te': te, 'n_tok': len(ts),
            'onset': bad[0] if bad else None, 'sents': sents, 'sent_of': sent_of,
            'ends': {sn['t1'] for sn in sents}}

REFS = {}
def to_items(rs):
    out = []
    for r in rs:
        REFS.setdefault(r['source_id'], build_reference(sources[r['source_id']]))
        out.append(prep(r))
    return out

dev_items, val_items = to_items(dev), to_items(val)
for name, it in [('dev', dev_items), ('검증', val_items)]:
    n_h = sum(x['onset'] is not None for x in it)
    print(f'{name}: 응답 {len(it)} (환각 {n_h}, 없음 {len(it)-n_h}) | 토큰 {sum(x["n_tok"] for x in it):,} | '
          f'환각 없는 응답 토큰 {sum(x["n_tok"] for x in it if x["onset"] is None):,}')

In [ ]:
TRAIN_ITEMS = to_items(train)
ITEMS = {f'tr-{x["id"]}': x for x in TRAIN_ITEMS}
LABELS = {k: {int(i): v for i, v in d.items()} for k, d in json.load(open(os.path.join(R4B_DIR, 'labels_train.json'))).items()}
print(f'학습 재판정 {len(LABELS):,}/{len(ITEMS):,}')

In [ ]:
def content_starts(x):
    t = x['text']
    cs = [a + (len(t[a:b]) - len(t[a:b].lstrip())) for a, b in zip(x['ts'], x['te'])]
    return [c if c < b else a for c, a, b in zip(cs, x['ts'], x['te'])]

def locate(x, sn, phrase, cs):
    seg = x['text'][sn['a']:sn['b']]
    ph = (phrase or '').strip()
    k = seg.find(ph) if ph else -1
    if k < 0 and ph: k = seg.lower().find(ph.lower())
    if k < 0: return sn['t0']
    a, b = sn['a'] + k, sn['a'] + k + len(ph)
    for t in range(sn['t0'], sn['t1'] + 1):
        if min(x['te'][t], b) > max(cs[t], a): return t
    return sn['t0']

ADD_ONSET, flip, earlier = {}, 0, 0
for cid, x in ITEMS.items():
    labs = LABELS.get(cid)
    if not labs: continue
    cs, add = None, []
    for i, sn in enumerate(x['sents']):
        if sn['y'] == 0 and labs[i + 1]['label'] == 'H':
            cs = cs or content_starts(x)
            t = locate(x, sn, labs[i + 1]['phrase'], cs); ADD_ONSET[(x['id'], i)] = t; add.append(t)
    if add:
        if x['onset'] is None: flip += 1
        elif min(add) < x['onset']: earlier += 1
n_clean = sum(x['onset'] is None for x in TRAIN_ITEMS); n_hall = len(TRAIN_ITEMS) - n_clean
print(f'추가된 환각 문장 {len(ADD_ONSET):,} | 멀쩡 → 환각 {flip}/{n_clean} ({flip / max(n_clean, 1):.0%}) | 더 이른 환각 {earlier}/{n_hall} ({earlier / max(n_hall, 1):.0%})')

## 3. 학습 판정 지점 (F1과 같음) + 최신 LLM 학습 응답

In [ ]:
def points(item, s):
    pts = []
    for sn in item['sents']:
        for t in range(sn['t0'], sn['t1'] + 1):
            k = t - sn['t0'] + 1
            if (s is not None and k % s == 0) or t == sn['t1']:
                pts.append((t, item['text'][sn['a']:min(item['te'][t], sn['b'])].strip()))
    return pts


WIN_TOK = 10
train_items = TRAIN_ITEMS
by_tr = {str(r['id']): r for r in train}
TR, P_KEY, P_TXT, P_CAT = [], [], [], []
for x in train_items:
    text = x['text']
    spans = [(l['start'], l['end']) for l in (by_tr[x['id']].get('labels') or [])]
    cs = [a + (len(text[a:b]) - len(text[a:b].lstrip())) if text[a:b].strip() else a for a, b in zip(x['ts'], x['te'])]
    def s_onset(sn):                      # E2와 같다: 이 문장 안에서 환각 구간과 처음 겹치는 토큰
        for t in range(sn['t0'], sn['t1'] + 1):
            if any(min(x['te'][t], b) > max(cs[t], a) for a, b in spans): return t
        return None
    sid = {sn['t0']: i for i, sn in enumerate(x['sents'])}
    son = [s_onset(sn) for sn in x['sents']]
    son = [o if o is not None else ADD_ONSET.get((x['id'], i)) for i, o in enumerate(son)]   # R4b 보정 학습 정답
    groups = {}
    for t, txt in points(x, 2):
        if not txt: continue
        si = sid[x['sent_of'][t]]; o = son[si]
        if o is None: cat = 'neg'
        elif t < o: cat = 'pre'
        elif t <= o + WIN_TOK: cat = 'win'
        else: cat = 'post'
        g = len(P_TXT); P_KEY.append(x['ref_key']); P_TXT.append(txt); P_CAT.append(cat)
        if cat in ('win', 'post'): groups.setdefault(si, []).append(g)
    gl = []
    for si, lst in groups.items():
        win = [g for g in lst if P_CAT[g] == 'win']
        if not win:                       # 창 안에 지점이 없으면 시작 이후 첫 지점을 창으로
            P_CAT[lst[0]] = 'win'; win = [lst[0]]
        gl.append(win)
    TR.append({'groups': gl})
P_CAT = np.array(P_CAT)
print(f'학습 응답 {len(TR)} | 판정 지점 {len(P_TXT):,} | 환각 문장(다중 사례 단위) {sum(len(r["groups"]) for r in TR):,} |',
      pd.Series(P_CAT).value_counts().to_dict())
assert sum(len(r['groups']) for r in TR) > 0

In [ ]:
K2_LABEL_SRC = 'SYSTEM = \'\'\'You are an expert annotator for RAGTruth, a benchmark of hallucinations in retrieval-augmented generation.\nYou receive a REFERENCE that a language model was given, and the model\'s RESPONSE split into numbered sentences.\nLabel EVERY sentence against the REFERENCE only (not world knowledge).\n\nA sentence contains a hallucination if any part of it is:\n- Evident conflict: clearly contradicts the reference (wrong fact, number, name, date, negation).\n- Subtle conflict: deviates from the reference in a way that changes meaning (altered term, scope, time, attribution).\n- Evident baseless info: introduces information that is not in the reference, even if it may be true in the real world.\n- Subtle baseless info: adds unsupported inference, opinion, or embellishment (e.g. "popular", "cozy", "family-friendly") that the reference does not support.\n\nNot hallucinations: faithful paraphrase; reasonable summarization; statements that the reference lacks the information or declining to answer;\nmeta phrases such as "Based on the passages,". For Data2txt (business data + user reviews), claims supported by the structured data or by what reviewers say are supported.\n\nLabels: "S" = no hallucination. "H" = contains a hallucination. "A" = ambiguous, careful annotators could reasonably disagree.\nFor every H or A sentence also give: "type" (one of: evident_conflict, subtle_conflict, evident_baseless, subtle_baseless),\n"phrase" (the shortest span copied EXACTLY, character for character, from that sentence that is hallucinated), and "reason" (at most 20 words).\n\nReturn ONLY a JSON object, no other text:\n{"sentences": [{"i": 1, "label": "S"}, {"i": 2, "label": "H", "type": "evident_baseless", "phrase": "...", "reason": "..."}]}\nwith exactly one entry per numbered sentence, in order.\'\'\'\n\nTASK_DESC = {\'QA\': \'QA — answer the question using the passages\',\n             \'Summary\': \'Summary — summarize the news article\',\n             \'Data2txt\': \'Data2txt — write an overview of a business from structured data and user reviews\'}\n\ndef sentences(text):\n    # (시작, 끝, 문장) — 판정 지점(points_of)과 같은 pysbd 분할\n    out = []\n    for sp in SEG.segment(text):\n        if not sp.sent.strip(): continue\n        a = sp.start + (len(sp.sent) - len(sp.sent.lstrip()))\n        out.append((a, sp.start + len(sp.sent.rstrip()), sp.sent.strip()))\n    return out\n\ndef label_msg(c, text):\n    ss = sentences(text)\n    lines = \'\\n\'.join(f\'[{i + 1}] {s}\' for i, (_, _, s) in enumerate(ss))\n    return f"TASK: {TASK_DESC[c[\'task\']]}\\n\\nREFERENCE:\\n<<<\\n{c[\'ref_label\']}\\n>>>\\n\\nRESPONSE ({len(ss)} numbered sentences):\\n{lines}"\n\ndef label_params(c, text):\n    # R4와 같은 요청 (모델·max_tokens·지시문)\n    return {\'model\': LABELER, \'max_tokens\': 4096, \'system\': SYSTEM, \'messages\': [{\'role\': \'user\', \'content\': label_msg(c, text)}]}\n\ndef parse(text, n):\n    try:\n        d = json.loads(text[text.find(\'{\'): text.rfind(\'}\') + 1])\n        labs = {}\n        for e in d.get(\'sentences\', []):\n            i, lab = int(e.get(\'i\', 0)), str(e.get(\'label\', \'\')).strip().upper()[:1]\n            if 1 <= i <= n and lab and lab in \'SHA\':\n                labs[i] = {\'label\': lab, \'type\': e.get(\'type\', \'\'), \'phrase\': e.get(\'phrase\', \'\') or \'\', \'reason\': e.get(\'reason\', \'\')}\n        return labs, len(labs) == n\n    except Exception:\n        return {}, False\n'

# K2a 최신 LLM 학습 응답(Sonnet 문장 정답)을 학습 판정 지점에 더한다. 지점·범주 규칙은 위 셀(F1)과 같다:
# H 문장은 정답 문구 위치(locate, R4b ADD와 같음)부터 환각, 나머지 문장은 음성. 근거는 판정 때와 같게(QA = 검색된 조각)
import pickle
def _lj(p): return json.load(open(p))
K2T = {n: _lj(os.path.join(K2_DIR, f'{n}_train.json')) for n in ['cases', 'gen', 'label']}
_hold = {x['ref_key'] for k in ['calL', 'devL'] for x in pickle.load(open(os.path.join(R0_DIR, 'r0_items.pkl'), 'rb'))['items'][k]}
assert not _hold & {c['sid'] for c in K2T['cases'].values()}, 'K2 학습 응답에 cal-L·dev-L 근거가 섞임'
_K2L = {'SEG': seg, 'json': json, 're': __import__('re')}
exec(K2_LABEL_SRC, _K2L)                           # sentences, parse (M1과 같은 코드, 이 노트북 이름과 분리)
K2_MAX_RATIO = 1.0                                  # 구현 보충(SSOT §6 K2): 최신 LLM 지점 수 ≤ 기존 학습 지점 수 (응답 단위 무작위, 시드 2032)

def k2_item(cid, c, text):
    key = c['sid'] if c['flow'] == 'direct' else f'k2q:{cid}'
    x = prep({'response': text, 'id': cid, 'task_type': c['task'], 'model': c['gen'], 'source_id': key, 'labels': []})
    return key, x

def k2_points(x, labs, text):
    ss = _K2L['sentences'](text)
    starts = [a for a, _, _ in ss]
    cs = content_starts(x)
    son = []
    for sn in x['sents']:
        i = next((k for k, a in enumerate(starts) if sn['a'] <= a < sn['b']), None)
        lab = labs.get(i + 1) if i is not None else None
        son.append(locate(x, sn, lab.get('phrase'), cs) if lab and lab['label'] == 'H' else None)
    sid = {sn['t0']: i for i, sn in enumerate(x['sents'])}
    out = []
    for t, txt in points(x, 2):
        if not txt: continue
        si = sid[x['sent_of'][t]]; o = son[si]
        cat = 'neg' if o is None else 'pre' if t < o else 'win' if t <= o + WIN_TOK else 'post'
        out.append((si, txt, cat))
    return out, sum(o is not None for o in son)

K2_RESP, K2_NH, K2_NPARSE, K2_NVALID = [], 0, 0, 0
for cid, c in K2T['cases'].items():
    g = K2T['gen'].get(cid)
    if not g or g.get('stop') == 'refusal' or not g['text'].strip() or not _K2L['sentences'](g['text']): continue
    K2_NVALID += 1
    raw = K2T['label'].get(cid)
    if raw is None: continue
    labs, ok = _K2L['parse'](raw['text'], len(_K2L['sentences'](g['text'])))
    if not ok: continue
    K2_NPARSE += 1
    key, x = k2_item(cid, c, g['text'])
    pts, nh = k2_points(x, labs, g['text'])
    if not pts: continue
    K2_NH += nh
    K2_RESP.append({'cid': cid, 'key': key, 'ref': c['ref_judge'], 'pts': pts, 'nh': nh, 'gen': c['gen'], 'task': c['task']})
_n_orig, _n_k2 = len(P_TXT), sum(len(r['pts']) for r in K2_RESP)
_order = list(range(len(K2_RESP))); random.Random(2032).shuffle(_order)
K2_USE, _tot = [], 0
for k in _order:
    if _tot + len(K2_RESP[k]['pts']) > K2_MAX_RATIO * _n_orig: continue
    K2_USE.append(K2_RESP[k]); _tot += len(K2_RESP[k]['pts'])
P_CAT = list(P_CAT); _n0 = len(P_TXT)
for r in K2_USE:
    REFS.setdefault(r['key'], r['ref'])
    groups = {}
    for si, txt, cat in r['pts']:
        g = len(P_TXT); P_KEY.append(r['key']); P_TXT.append(txt); P_CAT.append(cat)
        if cat in ('win', 'post'): groups.setdefault(si, []).append(g)
    gl = []
    for si, lst in groups.items():
        win = [g for g in lst if P_CAT[g] == 'win']
        if not win: P_CAT[lst[0]] = 'win'; win = [lst[0]]
        gl.append(win)
    TR.append({'groups': gl})
P_CAT = np.array(P_CAT)
K2_STATS = {'valid': K2_NVALID, 'parsed': K2_NPARSE, 'parse_ratio': K2_NPARSE / max(K2_NVALID, 1), 'H_sent': int(K2_NH),
            'resp_all': len(K2_RESP), 'pts_all': _n_k2, 'resp_used': len(K2_USE), 'pts_used': len(P_TXT) - _n0,
            'H_sent_used': int(sum(r['nh'] for r in K2_USE)), 'hal_resp_used': int(sum(r['nh'] > 0 for r in K2_USE)), 'pts_orig': _n_orig}
print('최신 LLM 학습 응답:', K2_STATS)
print('추가 지점 범주', pd.Series(P_CAT[_n0:]).value_counts().to_dict(), '| 전체', pd.Series(P_CAT).value_counts().to_dict())

## 4. 근거 검색·평가 세트·평가 함수 (F1과 같음)

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

retriever = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)
K, MAX_LEN, CHUNK_WORDS, BS = 8, 512, 40, 32

chunk_cache = {}
def get_chunks(key):
    if key not in chunk_cache:
        w = REFS[key].split()
        ch = [' '.join(w[i:i + CHUNK_WORDS]) for i in range(0, max(len(w), 1), CHUNK_WORDS)] or [REFS[key]]
        chunk_cache[key] = (ch, retriever.encode(ch, convert_to_tensor=True, normalize_embeddings=True,
                                                 batch_size=128, show_progress_bar=False))
    return chunk_cache[key]

def contexts(keys, texts, bs=256):
    # 텍스트마다 근거 top-K 조각을 원문 순서로 이어 붙인 문자열
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            top = torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()
            out.append(' ... '.join(ch[t] for t in sorted(top)))
    return out


# R0 평가 세트(cal-L, dev-L)와 기존 판정기 점수
import pickle
R0P = pickle.load(open(os.path.join(R0_DIR, 'r0_items.pkl'), 'rb'))
Z0 = dict(np.load(os.path.join(R0_DIR, 'r0_scores.npz')))
CAL, DEV = R0P['items']['calL'], R0P['items']['devL']
Q_KEYS, Q_TEXTS = R0P['q_keys'], R0P['q_texts']
for k in set(Q_KEYS):
    REFS.setdefault(k, build_reference(sources[k]))

@torch.no_grad()
def topk_idx(keys, texts, bs=512):
    out = []
    for i in range(0, len(texts), bs):
        embs = retriever.encode(texts[i:i + bs], convert_to_tensor=True, normalize_embeddings=True,
                                batch_size=bs, show_progress_bar=False)
        for key, e in zip(keys[i:i + bs], embs):
            ch, ce = get_chunks(key)
            out.append(sorted(torch.topk(ce @ e, min(K, ce.shape[0])).indices.tolist()))
    return out
ctx_of = lambda key, idx: ' ... '.join(get_chunks(key)[0][j] for j in idx)

t0 = time.perf_counter()
P_IDX = topk_idx(P_KEY, P_TXT)
print(f'학습 지점 근거 검색 {time.perf_counter()-t0:.0f}초')

# 평가 질의: cal-L·dev-L 지점만 (R0와 같은 질의 번호)
EVAL_Q = sorted({q for x in CAL + DEV for _, q in x['pts2']})
t0 = time.perf_counter()
EQ_IDX = dict(zip(EVAL_Q, topk_idx([Q_KEYS[q] for q in EVAL_Q], [Q_TEXTS[q] for q in EVAL_Q])))
print(f'평가 질의 {len(EVAL_Q):,}개 근거 검색 {time.perf_counter()-t0:.0f}초')

# 체크포인트 선택용 cal-sub (cal-L에서 300응답 고정)
CALSUB = random.Random(1).sample(CAL, min(300, len(CAL)))

In [ ]:
# R4 보정 정답 C1 (Opus 재판정)을 cal-L·dev-L에 적용한 사본
CORR = json.load(open(os.path.join(R4_DIR, 'r4_corrected_labels.json')))['C1']
def apply_corr(items, split):
    out = []
    for x in items:
        c = CORR[split].get(x['id'])
        y = copy.copy(x); y['sents'] = [dict(sn) for sn in x['sents']]
        if c is not None:
            y['onset'] = c['onset']
            for i in c['added']: y['sents'][i - 1]['y'] = 1
        out.append(y)
    return out
CAL_C, DEV_C = apply_corr(CAL, 'calL'), apply_corr(DEV, 'devL')
LAB_SETS = {'orig': (CAL, DEV), 'C1': (CAL_C, DEV_C)}
for k, (c, d) in LAB_SETS.items():
    print(f"{k}: cal-L 환각 {sum(x['onset'] is not None for x in c)}, dev-L 환각 {sum(x['onset'] is not None for x in d)}")
# 체크포인트 선택은 보정 정답 기준 cal-sub (같은 300응답)
_byid = {x['id']: x for x in CAL_C}
CALSUB = [_byid[x['id']] for x in CALSUB]

In [ ]:
SC = dict(Z0)

def series(x, scorer):
    m, k = scorer.split('|')
    S_ = SC[m]
    if k == 'end':
        return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends']]
    if k == 'ew2':
        out, s = [], None
        for t, q in x['pts2']:
            v = float(S_[q]); s = v if s is None else 0.5 * v + 0.5 * s
            out.append((t, s))
        return out
    s = int(k)
    return [(t, S_[q]) for t, q in x['pts2'] if t in x['ends'] or (t - x['sent_of'][t] + 1) % s == 0]

In [ ]:
def calibrate(items, scorer, arl0):
    clean = [x for x in items if x['onset'] is None]
    n_tok = sum(x['n_tok'] for x in clean)
    sc = np.sort(np.concatenate([[s for _, s in series(x, scorer)] for x in clean]))[::-1]
    n_alarm = max(1, int(n_tok / arl0))
    return float(sc[n_alarm - 1]) + 1e-6 if n_alarm <= len(sc) else float(sc[-1])

def evaluate(items, scorer, thr):
    # 엄격 기준(주 지표, 선행 연구와 같은 정의): 응답의 **첫 경보**가 발생 θ 이후여야 탐지로 인정.
    #   θ 전에 경보가 울리면 그 응답은 오경보로 끝난 것(미탐지)으로 본다.
    # 관대 기준(참고): θ 전 경보는 무시하고 θ 이후 첫 경보를 탐지로 인정 — v1의 주 지표였으나 재현율이 부풀려짐.
    clean = [x for x in items if x['onset'] is None]
    hall = [x for x in items if x['onset'] is not None]
    alarms = sum(sum(s >= thr for _, s in series(x, scorer)) for x in clean)
    arl = sum(x['n_tok'] for x in clean) / max(alarms, 1)
    clean_flag = np.mean([any(s >= thr for _, s in series(x, scorer)) for x in clean])
    st_det, st_d, st_ch, cedd, le_det, pre = 0, [], [], [], 0, 0
    for x in hall:
        ser = series(x, scorer)
        first = next((t for t, s in ser if s >= thr), None)
        le_hit = next((t for t, s in ser if t >= x['onset'] and s >= thr), None)
        le_det += le_hit is not None
        pre += first is not None and first < x['onset']
        if first is not None and first >= x['onset']:
            st_det += 1; st_d.append(first - x['onset']); cedd.append(first - x['onset'])
            st_ch.append(x['te'][first] - x['ts'][x['onset']])
        else:
            cedd.append(x['n_tok'] - x['onset'])
    return {'realized_ARL0_dev': arl, 'clean_resp_flagged': clean_flag,
            'recall': st_det / len(hall), 'delay_tok_mean': np.mean(st_d) if st_d else np.nan,
            'delay_tok_median': np.median(st_d) if st_d else np.nan,
            'delay_chars_median': np.median(st_ch) if st_ch else np.nan,
            'censored_EDD_tok': np.mean(cedd), 'pre_onset_alarm_rate': pre / len(hall),
            'recall_lenient': le_det / len(hall),
            'calls_per_resp': np.mean([len(series(x, scorer)) for x in items])}

def calibrate_flag(items, scorer, rate):
    # 서비스 기준 보정: 환각 없는 응답 중 표시되는 비율이 rate 이하가 되는 임계값 (응답별 최대 점수의 분위수)
    mx = np.array([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    return float(np.quantile(mx, 1 - rate)) + 1e-6

## 5. 학습 (F1 base와 같은 루프, 시드 42) — 저장 위치 `k2/`

In [ ]:
KEEP_WEIGHTS = {42}
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F, shutil

BACKBONES = {'mini': ('cross-encoder/nli-MiniLM2-L6-H768', 2e-5),
             'base': ('cross-encoder/nli-deberta-v3-base', 2e-5),
             'large': ('lytang/MiniCheck-DeBERTa-v3-Large', 1e-5)}
BATCH_SEQ, MICRO, N_EVALS = 64, 16, 4
BF16 = DEVICE == 'cuda' and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
if not BF16: print('⚠️ bf16 미지원 GPU — DeBERTa는 fp16에서 불안정할 수 있음 (A100/L4 권장)')

def enc_ids(tok, idxs):
    return tok([ctx_of(P_KEY[g], P_IDX[g]) for g in idxs], [P_TXT[g] for g in idxs], truncation='only_first',
               max_length=MAX_LEN, padding=True, return_tensors='pt').to(DEVICE)

@torch.no_grad()
def score_queries(tok, model, qs, bs=128):
    model.eval(); out = np.zeros(len(qs), dtype=np.float32)
    order = sorted(range(len(qs)), key=lambda i: len(Q_TEXTS[qs[i]]))
    for i in range(0, len(order), bs):
        b = [qs[j] for j in order[i:i + bs]]
        enc = tok([ctx_of(Q_KEYS[q], EQ_IDX[q]) for q in b], [Q_TEXTS[q] for q in b], truncation='only_first',
                  max_length=MAX_LEN, padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
            out[order[i:i + bs]] = model(**enc).logits[:, 0].float().cpu().numpy()
    model.train()
    return out

CALSUB_Q = sorted({q for x in CALSUB for _, q in x['pts2']})
def calsub_metric(tok, model):
    v = score_queries(tok, model, CALSUB_Q)
    SC['_cur'] = np.zeros(len(Q_KEYS), dtype=np.float32); SC['_cur'][CALSUB_Q] = v
    r4 = evaluate(CALSUB, '_cur|4', calibrate(CALSUB, '_cur|4', 100))
    r2 = evaluate(CALSUB, '_cur|2', calibrate(CALSUB, '_cur|2', 100))
    d4 = r4['delay_tok_median']
    return float((r4['recall'] + r2['recall']) / 2), float(d4) if d4 == d4 else 1e9

Y_ALL = np.isin(P_CAT, ['win', 'post']).astype(np.float32)          # V0b: 문장 단위 시작 이후 = 1

def load_init(name):
    path, _ = BACKBONES[name]
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForSequenceClassification.from_pretrained(path, num_labels=1, ignore_mismatched_sizes=True).to(DEVICE)
    return tok, model

def train_backbone(name, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tok, model = load_init(name)
    lr = BACKBONES[name][1]
    order = list(range(len(P_TXT))); random.Random(seed).shuffle(order)
    n_steps = max(1, len(order) // BATCH_SEQ)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * n_steps), n_steps)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda' and not BF16))
    best, d0 = calsub_metric(tok, model)
    best_key, best_state, hist = (d0 <= 9, best), copy.deepcopy(model.state_dict()), [(0, best, d0)]
    print(f'  [{name} s{seed}] 시작 cal-sub 재현율 {best:.3f}, 지연 {d0} | {n_steps}단계, 학습률 {lr}')
    eval_at = {int(n_steps * (i + 1) / N_EVALS) for i in range(N_EVALS)}
    model.train(); t0 = time.perf_counter()
    for step in range(1, n_steps + 1):
        batch = order[(step - 1) * BATCH_SEQ: step * BATCH_SEQ]
        opt.zero_grad(); tot = 0.0
        for k in range(0, len(batch), MICRO):
            mb = batch[k:k + MICRO]
            with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=(DEVICE == 'cuda')):
                z = model(**enc_ids(tok, mb)).logits[:, 0].float()
            loss = F.binary_cross_entropy_with_logits(z, torch.tensor(Y_ALL[mb], device=DEVICE), reduction='sum') / len(batch)
            scaler.scale(loss).backward(); tot += loss.item()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sch.step()
        if step in eval_at:
            va, dv = calsub_metric(tok, model); hist.append((step, va, dv))
            if (dv <= 9, va) > best_key: best_key, best, best_state = (dv <= 9, va), va, copy.deepcopy(model.state_dict())
            print(f'  [{name} s{seed}] 단계 {step}/{n_steps} | 손실 {tot:.3f} | cal-sub 재현율 {va:.3f}, 지연 {dv} | {(time.perf_counter()-t0)/60:.1f}분')
    model.load_state_dict(best_state)
    return tok, model, hist

def run(name, seed):
    out = os.path.join(R1_DIR, f'{name}_s{seed}')
    sp = os.path.join(out, 'scores.npy')
    if not os.path.exists(sp):
        has_w = any(os.path.exists(os.path.join(out, f)) for f in ['model.safetensors', 'pytorch_model.bin'])
        if os.path.exists(os.path.join(out, 'config.json')) and not has_w:
            print(f'  [{name} s{seed}] 저장이 끊긴 폴더 → 지우고 다시 학습'); shutil.rmtree(out)
        if has_w:
            tok = AutoTokenizer.from_pretrained(out); model = AutoModelForSequenceClassification.from_pretrained(out).to(DEVICE)
        else:
            tok, model, hist = train_backbone(name, seed)
            os.makedirs(out, exist_ok=True); tok.save_pretrained(out); model.save_pretrained(out)
            pd.DataFrame(hist, columns=['step', 'calsub_recall', 'calsub_delay4']).to_csv(os.path.join(out, 'hist.csv'), index=False)
        t0 = time.perf_counter()
        v = score_queries(tok, model, EVAL_Q)
        full = np.full(len(Q_KEYS), np.nan, dtype=np.float32); full[EVAL_Q] = v
        np.save(sp, full)
        if seed not in KEEP_WEIGHTS:                     # 드라이브 용량: 가중치는 시드 42만 남긴다
            for f in ['model.safetensors', 'pytorch_model.bin']:
                if os.path.exists(os.path.join(out, f)): os.remove(os.path.join(out, f))
        print(f'  [{name} s{seed}] 평가 질의 채점 {time.perf_counter()-t0:.0f}초')
        del model; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    SC[f'{name}_s{seed}'] = np.load(sp)
    print(f'[{name} s{seed}] 준비 완료')

In [ ]:
LOCAL = '/content/k2_local'
os.makedirs(LOCAL, exist_ok=True)
WFILES = ['model.safetensors', 'pytorch_model.bin']
NAMES = ['base', 'mini']
BACKBONE_OF = {'base': 'cross-encoder/nli-deberta-v3-base (R1c 방식, 사전학습 NLI에서 시작)',
               'mini': 'cross-encoder/nli-MiniLM2-L6-H768 → student_vark에서 시작 (R1b V0b 방식)'}
DRIVE_OK = {}

def has_weights(d):
    return os.path.isdir(d) and any(os.path.exists(os.path.join(d, f)) and os.path.getsize(os.path.join(d, f)) > 0 for f in WFILES)

def sizes(d):
    return {f: os.path.getsize(os.path.join(d, f)) for f in sorted(os.listdir(d)) if os.path.isfile(os.path.join(d, f))}

def to_drive(name):
    src, dst = os.path.join(LOCAL, f'judge_{name}'), os.path.join(FINAL_DIR, f'judge_{name}')
    try:
        shutil.copytree(src, dst, dirs_exist_ok=True)
        ok = sizes(src) == {f: v for f, v in sizes(dst).items() if f in sizes(src)}
    except OSError as e:
        print(f'  [{name}] 드라이브 복사 실패: {e}'); ok = False
    DRIVE_OK[name] = ok
    print(f'  [{name}] 드라이브 저장 {"확인" if ok else "실패 — 마지막 셀 zip을 꼭 내려받을 것"} ({sum(sizes(src).values()) / 2**20:.0f}MB)')

def load_judge(name):
    d = os.path.join(LOCAL, f'judge_{name}')
    tok = AutoTokenizer.from_pretrained(d)
    model = AutoModelForSequenceClassification.from_pretrained(d).float().to(DEVICE)   # fp16 저장 → fp32로 올림 (게이트웨이와 같게)
    return tok, model

def finalize(name):
    d, dd = os.path.join(LOCAL, f'judge_{name}'), os.path.join(FINAL_DIR, f'judge_{name}')
    if not has_weights(d) and has_weights(dd):
        shutil.copytree(dd, d, dirs_exist_ok=True); print(f'  [{name}] 드라이브 저장본 사용')
    if not has_weights(d):
        t0 = time.perf_counter()
        if name == 'base':
            tok, model, hist = train_backbone('base', 42)
        else:
            tok, model, _, hist = train_variant('V0b', 42)
        os.makedirs(d, exist_ok=True)
        tok.save_pretrained(d); model.half().save_pretrained(d)
        pd.DataFrame(hist, columns=['step', 'calsub_recall', 'calsub_delay4']).to_csv(os.path.join(d, 'hist.csv'), index=False)
        print(f'  [{name}] 학습 {(time.perf_counter() - t0) / 60:.0f}분, 로컬 저장')
        del model; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
        to_drive(name)
    sp, spd = os.path.join(LOCAL, f'scores_{name}.npy'), os.path.join(FINAL_DIR, f'scores_{name}.npy')
    if not os.path.exists(sp) and os.path.exists(spd): shutil.copy(spd, sp)
    if not os.path.exists(sp):
        tok, model = load_judge(name)
        t0 = time.perf_counter()
        v = score_queries(tok, model, EVAL_Q)
        full = np.full(len(Q_KEYS), np.nan, dtype=np.float32); full[EVAL_Q] = v
        np.save(sp, full)
        try: shutil.copy(sp, spd)
        except OSError as e: print('  점수 드라이브 저장 실패', e)
        print(f'  [{name}] 평가 질의 {len(EVAL_Q):,}개 채점 {time.perf_counter() - t0:.0f}초')
        del model; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()
    SC[f'F_{name}'] = np.load(sp)
    DRIVE_OK.setdefault(name, has_weights(dd))
    print(f'[{name}] 준비 완료')

for name in ['base']:
    finalize(name)
for d in [os.path.join(LOCAL, 'judge_base'), os.path.join(K2_DIR, 'judge_base')]:
    try: shutil.copy(os.path.join(OLD_DIR, 'judge_base', 'judge_config.json'), os.path.join(d, 'judge_config.json'))
    except OSError as e: print('설정 파일 복사 실패', d, e)

## 6. 새 조합과 K2-3 (RAGTruth cal-L, 임계값 A)

In [ ]:
_tr = {str(r['id']): r for r in responses if r['split'] == 'train'}
assert all(x['id'] in _tr for x in CAL + DEV)
RESP_TEXT = {x['id']: _tr[x['id']]['response'] for x in CAL + DEV}

ALPHA = 0.10
def conformal_flag(items, scorer, alpha=ALPHA):
    mx = np.sort([max([s for _, s in series(x, scorer)] or [-1e9]) for x in items if x['onset'] is None])
    n = len(mx); k = int(np.ceil((n + 1) * (1 - alpha)))
    return float(mx[k - 1]) + 1e-6 if k <= n else float('inf')

def detect_vec(items, scorer, thr):
    # 환각 응답마다 엄격 탐지 여부(첫 경보가 발생 이후)와 지연
    det, dl = [], []
    for x in items:
        if x['onset'] is None: continue
        first = next((t for t, s in series(x, scorer) if s >= thr), None)
        ok = first is not None and first >= x['onset']
        det.append(ok); dl.append(first - x['onset'] if ok else np.nan)
    return np.array(det, dtype=float), np.array(dl)

In [ ]:
# I2 판정 입력 = (근거 전체, 응답 처음부터 판정 지점 토큰까지). 판정 지점·정답은 F1과 같다.
MAX_LEN2 = 2048                    # L2: RAGTruth train 근거+응답 p90 1,182 → 2,048
N_TRAIN = 80000                    # 학습 지점 표본 (F1은 모든 지점 약 25만 개를 1에폭 — 긴 입력이라 일부만)

def need_point(x, t):
    # 평가에 쓰는 지점만 채점: 문장 끝 + 4토큰 간격 (8·16토큰 간격은 그 부분집합)
    return t in x['ends'] or (t - x['sent_of'][t] + 1) % 4 == 0

def build_eval_points(items, resp_text):
    # 평가 지점마다 새 번호. R0 번호는 (근거, 문장 앞부분)으로 묶여 응답 앞부분이 다른 지점을 구분하지 못한다
    keys, texts, need, MAP = [], [], [], {}
    for x in items:
        text = resp_text[x['id']]
        assert len(text) >= x['te'][-1], x['id']
        for t, _ in x['pts2']:
            MAP[(x['id'], t)] = len(keys)
            keys.append(x['ref_key']); texts.append(text[:x['te'][t]]); need.append(need_point(x, t))
    return keys, texts, np.array(need), MAP

def remap(items, MAP):
    out = []
    for x in items:
        y = copy.copy(x); y['pts2'] = [(t, MAP[(x['id'], t)]) for t, _ in x['pts2']]
        out.append(y)
    return out


def to_E(items, name, MAP, n):
    # 기존 질의 번호 점수(F1 base)를 I2 평가 지점 번호로 옮긴다
    a = np.full(n, np.nan, dtype=np.float32)
    for x in items:
        for t, q in x['pts2']:
            if (x['id'], t) in MAP: a[MAP[(x['id'], t)]] = SC[name][q]
    return a

def clean_stats(items, scorer):
    # 표준화 기준: 멀쩡한 응답(cal-L 원래 정답)의 판정 지점 점수 평균·표준편차
    v = np.concatenate([[s for _, s in series(x, scorer)] for x in items if x['onset'] is None])
    return float(np.mean(v)), float(np.std(v))

def compare_e(cal, a, b, kind='8', task=None, n_boot=2000, seed=2027):
    # a(후보) − b(기준). 임계값은 cal 전체로 등각 서비스 10%(서비스와 같은 단일 임계값), task가 있으면 그 과제 환각 응답만 비교
    ta, tb = conformal_flag(cal, f'{a}|{kind}'), conformal_flag(cal, f'{b}|{kind}')
    sub = [x for x in cal if task is None or x['task_type'] == task]
    da, la = detect_vec(sub, f'{a}|{kind}', ta); db, lb = detect_vec(sub, f'{b}|{kind}', tb)
    rng = np.random.default_rng(seed); n = len(da)
    bt = [da[i].mean() - db[i].mean() for i in (rng.integers(0, n, n) for _ in range(n_boot))]
    return {'task': task or 'all', 'recall_new': float(da.mean()), 'recall_base': float(db.mean()), 'delta': float(da.mean() - db.mean()),
            'ci_lo': float(np.percentile(bt, 2.5)), 'ci_hi': float(np.percentile(bt, 97.5)),
            'delay_new': float(np.nanmedian(la)) if np.isfinite(la).any() else np.nan,
            'delay_base': float(np.nanmedian(lb)) if np.isfinite(lb).any() else np.nan,
            'n_hall': n, 'thr_new': ta, 'thr_base': tb}

def report_e(cal, dev, name, kind='8'):
    # dev-L 보고: cal 임계값으로 과제별 탐지율·지연과 멀쩡한 응답 표시율
    thr = conformal_flag(cal, f'{name}|{kind}'); out = []
    for task in [None] + TASKS:
        sub = [x for x in dev if task is None or x['task_type'] == task]
        det, dl = detect_vec(sub, f'{name}|{kind}', thr)
        flag = np.mean([any(s >= thr for _, s in series(x, f'{name}|{kind}')) for x in sub if x['onset'] is None])
        out.append({'model': name, 'task': task or 'all', 'recall': float(det.mean()),
                    'delay_med': float(np.nanmedian(dl)) if np.isfinite(dl).any() else np.nan, 'clean_flagged': float(flag)})
    return out

# 기존 조합(F1 base + I2)과 새 조합(K1 F1 + I2). 표준화·임계값은 I3와 같은 방식으로 cal-L에서만
SC['F_k2'] = SC['F_base']
SC['F_old'] = np.load(os.path.join(OLD_DIR, 'scores_base.npy'))
E_KEYS, E_TEXTS, E_NEED, E_MAP = build_eval_points(CAL + DEV, RESP_TEXT)
SC['I2'] = np.load(os.path.join(I2_DIR, 'scores_i2.npy'))
assert len(SC['I2']) == len(E_KEYS), 'I2 점수 길이가 평가 지점 수와 다름'
SC['Fb'] = to_E(CAL + DEV, 'F_old', E_MAP, len(E_KEYS))
SC['Kb'] = to_E(CAL + DEV, 'F_k2', E_MAP, len(E_KEYS))
LS = {lab: (remap(c, E_MAP), remap(d, E_MAP)) for lab, (c, d) in LAB_SETS.items()}
CALo, CALc = LS['orig'][0], LS['C1'][0]
_r = compare_e(CALo, 'I2', 'Fb')
assert abs(_r['thr_base'] - 3.968751) < 1e-3 and abs(_r['thr_new'] - 4.562501) < 1e-3, 'F1 base·I2 재현 실패 (I3와 다름)'
STATS = {m: clean_stats(CALo, f'{m}|8') for m in ['Fb', 'Kb', 'I2']}
Z = {m: (SC[m] - mu) / sd for m, (mu, sd) in STATS.items()}
SC['OLD'], SC['NEW'] = (Z['Fb'] + Z['I2']) / 2, (Z['Kb'] + Z['I2']) / 2
THR_OLD, THR_NEW = conformal_flag(CALo, 'OLD|8'), conformal_flag(CALo, 'NEW|8')
THR_KF = conformal_flag(CALo, 'Kb|8')
assert abs(THR_OLD - 2.991638706756592) < 1e-3, f'기존 조합 임계값 재현 실패 {THR_OLD}'
COMBO_OLD = {'F_mean': STATS['Fb'][0], 'F_std': STATS['Fb'][1], 'I_mean': STATS['I2'][0], 'I_std': STATS['I2'][1], 'threshold': THR_OLD}
COMBO_NEW = {'F_mean': STATS['Kb'][0], 'F_std': STATS['Kb'][1], 'I_mean': STATS['I2'][0], 'I_std': STATS['I2'][1], 'threshold': THR_NEW}
print('표준화 (평균, 표준편차):', {m: tuple(round(v, 4) for v in s) for m, s in STATS.items()})
print(f'임계값: 기존 조합 {THR_OLD:.4f} (재현) | 새 조합 {THR_NEW:.4f} | 새 F1 단독 {THR_KF:.4f}')

# K1-3 (안전, RAGTruth cal-L 8토큰 서비스 10%)
ROWS = []
for lab, cal in [('orig', CALo), ('C1', CALc)]:
    for task in [None] + TASKS:
        ROWS.append({'labels': lab, 'model': 'NEW', **compare_e(cal, 'NEW', 'OLD', task=task)})
        ROWS.append({'labels': lab, 'model': 'Kb', **compare_e(cal, 'Kb', 'Fb', task=task)})
cal_tab = pd.DataFrame(ROWS)
pd.set_option('display.width', 250)
print(cal_tab[['labels', 'model', 'task', 'recall_new', 'recall_base', 'delta', 'ci_lo', 'ci_hi', 'delay_new', 'delay_base', 'n_hall']].round(3).to_string(index=False))
_row = lambda lab: next(r for r in ROWS if r['labels'] == lab and r['task'] == 'all' and r['model'] == 'NEW')
K2_3 = bool(_row('orig')['delta'] >= -0.02 and _row('C1')['delta'] >= -0.02)
print(f"=== K2-3 (안전, RAGTruth, 임계값 A): 원래 정답 {_row('orig')['recall_base']:.3f} → {_row('orig')['recall_new']:.3f} (Δ {_row('orig')['delta']:+.3f}), "
      f"C1 Δ {_row('C1')['delta']:+.3f} (각각 ≥ −0.02) → {'충족' if K2_3 else '미달'} ===")
DEV_ROWS = []
for lab in ['orig', 'C1']:
    cal, dev = LS[lab]
    for m in ['Fb', 'Kb', 'OLD', 'NEW']:
        DEV_ROWS += [{'labels': lab, **r} for r in report_e(cal, dev, m)]
dev_tab = pd.DataFrame(DEV_ROWS)
print('dev-L (보고만)'); print(dev_tab.round(3).to_string(index=False))

## 7. 최신 LLM 세트 다시 판정 · 임계값 B

게이트웨이 `judge.py`로 새 F1 판정기를 불러, K2a·M1과 같은 판정 지점에서 F1 점수만 다시 낸다(I2 점수·지점 위치는 저장된 것 그대로).
먼저 기존 판정기로 저장 점수가 재현되는지 확인한다. 임계값 B는 cal-M 멀쩡한 응답의 최대 점수로 등각 보정(10%)한다.

In [ ]:
SRC = {'judge': '"""문장 판정기.\n\n입력 방식은 판정기 폴더의 `judge_config.json`이 정한다 (SPEC §4).\n  retrieval (F1 base): 근거 → 40단어 조각 → 임베딩 (요청당 1회, prepare)\n                       판정 텍스트 → 임베딩 → 유사한 조각 top-k (원문 순서) → 교차 인코더 [근거 조각, 문장 앞부분]\n  full      (I2식)    : 교차 인코더 [근거 전체, 응답 처음부터 판정 지점까지]. 근거 검색 없음\n점수는 logit(클수록 환각). 임계값은 재판정 간격(stride)별로 설정 파일의 값을 쓴다.\n"""\nimport json\nimport os\nimport threading\nimport time\nfrom dataclasses import dataclass\n\nimport torch\n\n\n@dataclass\nclass Verdict:\n    score: float          # logit, 클수록 환각\n    hallucinated: bool\n    judge_ms: float       # 검색 + 판정 시간\n\n\n@dataclass\nclass RefIndex:\n    chunks: list\n    emb: object           # torch.Tensor [n_chunks, d] (retrieval) / None (full)\n    prepare_ms: float\n    text: str = \'\'        # 근거 원문 (full 방식)\n\n\ndef chunk_text(text, n):\n    w = text.split()\n    return [\' \'.join(w[i:i + n]) for i in range(0, max(len(w), 1), n)] or [text]\n\n\ndef _stride_key(stride):\n    return \'end\' if not stride else str(stride)\n\n\ndef read_config(model_dir):\n    p = os.path.join(model_dir, \'judge_config.json\')\n    return json.load(open(p)) if os.path.isfile(p) else {}\n\n\ndef input_mode(cfg):\n    """설정 파일에서 입력 방식을 정한다. 명시(`input_mode`)가 없으면 I2 노트북 형식(retrieval: null)을 full로 본다."""\n    if cfg.get(\'input_mode\'):\n        return cfg[\'input_mode\']\n    inp = cfg.get(\'input\') or {}\n    return \'full\' if isinstance(inp, dict) and \'retrieval\' in inp and inp[\'retrieval\'] is None else \'retrieval\'\n\n\nclass StudentJudge:\n    def __init__(self, model_dir, embedder=\'sentence-transformers/all-MiniLM-L6-v2\', k=8, max_len=512,\n                 chunk_words=40, threshold=0.0, device=\'\', mode=\'retrieval\', thresholds=None, threshold_key=\'svc10_orig\',\n                 default_stride=0):\n        from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer\n        self.device = device or (\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        self.mode = mode\n        self.tok = AutoTokenizer.from_pretrained(model_dir)\n        cfg = AutoConfig.from_pretrained(model_dir)\n        kw = {}\n        if getattr(cfg, \'model_type\', \'\') == \'modernbert\':\n            # 새 transformers는 reference_compile을 모델 인자로 받지 않으므로 설정에 넣는다\n            if hasattr(cfg, \'reference_compile\'):\n                cfg.reference_compile = False\n            kw[\'attn_implementation\'] = \'sdpa\'\n        # 가중치가 fp16으로 저장돼 있어도 fp32로 올린다 (CPU 판정, 노트북 점수와 일치). GPU에서는 autocast가 반정밀도로 돌린다.\n        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir, config=cfg, **kw).float().to(self.device).eval()\n        if mode == \'retrieval\' and isinstance(embedder, str):\n            from sentence_transformers import SentenceTransformer\n            embedder = SentenceTransformer(embedder, device=self.device)\n        self.embedder = embedder\n        self.k, self.max_len, self.chunk_words, self.threshold = k, max_len, chunk_words, threshold\n        self.thresholds, self.threshold_key = thresholds or {}, threshold_key\n        self.default_stride = default_stride\n        self._lock = threading.Lock()     # 모델 1개를 여러 요청이 공유하므로 직렬화\n\n    def threshold_for(self, stride, key=None):\n        """재판정 간격별 임계값 (설정 파일 thresholds[간격][key]). 없으면 기본 임계값."""\n        return (self.thresholds.get(_stride_key(stride)) or {}).get(key or self.threshold_key, self.threshold)\n\n    def _embed(self, texts):\n        return self.embedder.encode(texts, convert_to_tensor=True, normalize_embeddings=True,\n                                    show_progress_bar=False)\n\n    def prepare(self, reference):\n        t0 = time.perf_counter()\n        if self.mode == \'full\':\n            return RefIndex([], None, (time.perf_counter() - t0) * 1000, reference)\n        chunks = chunk_text(reference, self.chunk_words)\n        with self._lock:\n            emb = self._embed(chunks)\n        return RefIndex(chunks, emb, (time.perf_counter() - t0) * 1000, reference)\n\n    def _pair(self, ref, text, prior):\n        if self.mode == \'full\':\n            return ref.text, (prior or \'\') + text\n        e = self._embed([text])[0]\n        top = torch.topk(ref.emb @ e, min(self.k, ref.emb.shape[0])).indices.tolist()\n        return \' ... \'.join(ref.chunks[i] for i in sorted(top)), text\n\n    @torch.no_grad()\n    def judge(self, ref, text, prior=\'\', threshold=None):\n        """text: 판정할 문장(또는 문장 앞부분, 앞뒤 공백 제거), prior: 응답에서 그 문장 앞까지의 원문."""\n        t0 = time.perf_counter()\n        with self._lock:\n            a, b = self._pair(ref, text, prior)\n            enc = self.tok([a], [b], truncation=\'only_first\', max_length=self.max_len,\n                           return_tensors=\'pt\').to(self.device)\n            with torch.autocast(self.device, enabled=(self.device == \'cuda\')):\n                score = float(self.model(**enc).logits.squeeze().float().cpu())\n            if self.device == \'cuda\':\n                torch.cuda.synchronize()\n        thr = self.threshold if threshold is None else threshold\n        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)\n\n\ndef load_judge(settings):\n    """판정기 폴더의 judge_config.json을 읽어 StudentJudge를 만든다. 환경 변수로 지정한 값이 설정 파일보다 우선한다."""\n    cfg = read_config(settings.judge_model_dir)\n    pick = lambda env, key, default: env if env is not None else cfg.get(key, default)\n    dflt = (cfg.get(\'default\') or {})\n    kind = dflt.get(\'kind\', \'end\')\n    return StudentJudge(settings.judge_model_dir,\n                        embedder=pick(settings.judge_embedder, \'embedder\', \'sentence-transformers/all-MiniLM-L6-v2\'),\n                        k=pick(settings.judge_k, \'k\', 8), max_len=pick(settings.judge_max_len, \'max_len\', 512),\n                        chunk_words=pick(settings.judge_chunk_words, \'chunk_words\', 40),\n                        threshold=pick(settings.judge_threshold, \'threshold\', 0.0), device=settings.judge_device,\n                        mode=input_mode(cfg),\n                        # 임계값을 환경 변수로 고정하면 간격별 임계값은 쓰지 않는다\n                        thresholds={} if settings.judge_threshold is not None else cfg.get(\'thresholds\', {}),\n                        threshold_key=dflt.get(\'threshold\', \'svc10_orig\'),\n                        default_stride=0 if kind == \'end\' else int(kind))\n\n\nclass LexicalJudge:\n    """모델 없이 파이프라인을 시험하기 위한 대체 판정기 (테스트·데모 전용).\n    근거에 없는 단어 비율을 점수로 쓴다. 판별력은 없다고 봐야 한다."""\n\n    default_stride = 0\n\n    def __init__(self, threshold=0.5, delay_ms=0.0, fail=None):\n        self.threshold, self.delay_ms = threshold, delay_ms\n        self.fail = fail          # 테스트용: 이 함수가 True를 돌려주는 판정에서 예외를 낸다\n\n    def threshold_for(self, stride, key=None):\n        return self.threshold\n\n    def prepare(self, reference):\n        return RefIndex([reference], {w.lower().strip(\'.,\') for w in reference.split()}, 0.0, reference)\n\n    def judge(self, ref, text, prior=\'\', threshold=None):\n        t0 = time.perf_counter()\n        if self.fail is not None and self.fail(text):\n            raise RuntimeError(\'judge failure (test)\')\n        if self.delay_ms:\n            time.sleep(self.delay_ms / 1000)\n        words = [w.lower().strip(\'.,\') for w in text.split() if w.strip(\'.,\')]\n        score = sum(w not in ref.emb for w in words) / max(len(words), 1)\n        thr = self.threshold if threshold is None else threshold\n        return Verdict(score, score >= thr, (time.perf_counter() - t0) * 1000)\n', 'store': '"""컬렉션 저장소 (SPEC §2.5, §2.6).\n\n관리자·개발자가 등록한 문서를 조각으로 나눠 임베딩과 함께 저장하고, 질문으로 검색한다.\n게이트웨이는 찾은 조각을 LLM 프롬프트에 넣고, **같은 조각을 판정 근거로** 쓴다(LLM이 본 근거 = 판정 근거).\n\n  Store 인터페이스\n    add(collection, text, title=\'\', doc_id=None) -> dict(doc_id, title, n_chunks, n_chars)\n    list(collection) -> [dict(doc_id, title, n_chunks, n_chars, created)]\n    delete(collection, doc_id) -> bool\n    collections() -> [dict(collection, n_docs)]\n    search(collection, query, k) -> [Evidence]       # 점수 내림차순\n\n처음 구현은 SQLite 파일 하나 + 임베딩 배열(코사인 유사도 전수 검색)이다. 규모가 커지면 같은 인터페이스로 벡터 DB로 바꾼다.\n임베더는 encode(texts) -> 정규화된 벡터 배열을 돌려주는 객체(sentence-transformers 호환)면 된다.\n"""\nimport sqlite3\nimport threading\nimport time\nimport uuid\nfrom dataclasses import asdict, dataclass\n\nimport numpy as np\n\nSCHEMA = \'\'\'\nCREATE TABLE IF NOT EXISTS documents (\n    collection TEXT NOT NULL, doc_id TEXT NOT NULL, title TEXT, text TEXT NOT NULL, created REAL NOT NULL,\n    PRIMARY KEY (collection, doc_id));\nCREATE TABLE IF NOT EXISTS chunks (\n    collection TEXT NOT NULL, doc_id TEXT NOT NULL, idx INTEGER NOT NULL, text TEXT NOT NULL, emb BLOB NOT NULL,\n    PRIMARY KEY (collection, doc_id, idx));\n\'\'\'\n\n\n@dataclass\nclass Evidence:\n    doc_id: str\n    title: str\n    chunk_idx: int\n    text: str\n    score: float\n\n    def to_dict(self):\n        return asdict(self)\n\n\ndef split_words(text, n, overlap=0):\n    """n단어 조각(겹침 overlap단어). 빈 문서는 조각 없음."""\n    w = text.split()\n    if not w:\n        return []\n    step = max(1, n - overlap)\n    return [\' \'.join(w[i:i + n]) for i in range(0, max(len(w) - overlap, 1), step)]\n\n\nclass SQLiteStore:\n    def __init__(self, path, embedder, chunk_words=120, overlap=20):\n        self.path, self.embedder = path, embedder\n        self.chunk_words, self.overlap = chunk_words, overlap\n        self._db = sqlite3.connect(path, check_same_thread=False)\n        self._db.executescript(SCHEMA)\n        self._lock = threading.Lock()\n        self._cache = {}          # collection -> (meta 리스트, 임베딩 행렬)\n\n    def _embed(self, texts):\n        v = self.embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False, convert_to_numpy=True)\n        return np.asarray(v, dtype=np.float32).reshape(len(texts), -1)\n\n    def add(self, collection, text, title=\'\', doc_id=None):\n        if not text or not text.strip():\n            raise ValueError(\'document text is empty\')\n        doc_id = doc_id or uuid.uuid4().hex[:16]\n        chunks = split_words(text, self.chunk_words, self.overlap)\n        emb = self._embed(chunks)\n        with self._lock, self._db:\n            self._db.execute(\'DELETE FROM chunks WHERE collection=? AND doc_id=?\', (collection, doc_id))   # 같은 id면 교체\n            self._db.execute(\'INSERT OR REPLACE INTO documents VALUES (?,?,?,?,?)\',\n                             (collection, doc_id, title or \'\', text, time.time()))\n            self._db.executemany(\'INSERT INTO chunks VALUES (?,?,?,?,?)\',\n                                 [(collection, doc_id, i, c, e.tobytes()) for i, (c, e) in enumerate(zip(chunks, emb))])\n            self._cache.pop(collection, None)\n        return {\'doc_id\': doc_id, \'title\': title or \'\', \'n_chunks\': len(chunks), \'n_chars\': len(text)}\n\n    def list(self, collection):\n        with self._lock:\n            rows = self._db.execute(\n                \'SELECT d.doc_id, d.title, length(d.text), d.created, \'\n                \'(SELECT count(*) FROM chunks c WHERE c.collection=d.collection AND c.doc_id=d.doc_id) \'\n                \'FROM documents d WHERE d.collection=? ORDER BY d.created\', (collection,)).fetchall()\n        return [{\'doc_id\': r[0], \'title\': r[1], \'n_chars\': r[2], \'created\': r[3], \'n_chunks\': r[4]} for r in rows]\n\n    def delete(self, collection, doc_id):\n        with self._lock, self._db:\n            n = self._db.execute(\'DELETE FROM documents WHERE collection=? AND doc_id=?\', (collection, doc_id)).rowcount\n            self._db.execute(\'DELETE FROM chunks WHERE collection=? AND doc_id=?\', (collection, doc_id))\n            self._cache.pop(collection, None)\n        return n > 0\n\n    def collections(self):\n        with self._lock:\n            rows = self._db.execute(\'SELECT collection, count(*) FROM documents GROUP BY collection ORDER BY collection\').fetchall()\n        return [{\'collection\': c, \'n_docs\': n} for c, n in rows]\n\n    def exists(self, collection):\n        with self._lock:\n            return self._db.execute(\'SELECT 1 FROM documents WHERE collection=? LIMIT 1\', (collection,)).fetchone() is not None\n\n    def _matrix(self, collection):\n        with self._lock:\n            if collection not in self._cache:\n                rows = self._db.execute(\n                    \'SELECT c.doc_id, d.title, c.idx, c.text, c.emb FROM chunks c JOIN documents d \'\n                    \'ON c.collection=d.collection AND c.doc_id=d.doc_id WHERE c.collection=? ORDER BY d.created, c.idx\',\n                    (collection,)).fetchall()\n                meta = [r[:4] for r in rows]\n                mat = np.stack([np.frombuffer(r[4], dtype=np.float32) for r in rows]) if rows else np.zeros((0, 1), np.float32)\n                self._cache[collection] = (meta, mat)\n            return self._cache[collection]\n\n    def search(self, collection, query, k):\n        meta, mat = self._matrix(collection)\n        if not meta or not query.strip():\n            return []\n        sims = mat @ self._embed([query])[0]\n        top = np.argsort(-sims)[:k]\n        return [Evidence(meta[i][0], meta[i][1], meta[i][2], meta[i][3], float(sims[i])) for i in top]\n\n    def close(self):\n        self._db.close()\n', 'context': '"""요청에서 근거 문서(RAG 컨텍스트)와 검증 옵션을 꺼낸다.\n\n우선순위\n1. 최상위 필드 `verification.context` (문자열 또는 문자열 리스트)\n   - OpenAI SDK에서는 `extra_body={"verification": {...}}` 로 보낸다.\n2. 메시지 본문 안의 `<context>...</context>` 태그 (RAG 서비스가 흔히 쓰는 형식)\n3. `verification.collection` — 게이트웨이에 등록된 컬렉션에서 검색 (app이 처리, SPEC §2.6)\n4. 모두 없으면 검증하지 않는다 (mode=off).\n\n`verification` 필드는 실제 LLM API가 모르는 필드이므로 전달 전에 제거한다.\n"""\nimport copy\nimport re\n\nCTX_TAG = re.compile(r\'<context>(.*?)</context>\', re.S | re.I)\nMODES = {\'pass\', \'hold\', \'off\'}\nACTIONS = {\'flag\', \'cut\'}\n\n\ndef _content_text(content):\n    if isinstance(content, str):\n        return content\n    if isinstance(content, list):          # [{"type":"text","text":...}, ...]\n        return \'\\n\'.join(p.get(\'text\', \'\') for p in content if isinstance(p, dict))\n    return \'\'\n\n\nRAG_INSTRUCTION = (\'Answer using only the reference documents below. \'\n                   \'If they do not contain the answer, say that you do not know.\')\n\n\ndef last_user_text(body):\n    users = [m for m in body.get(\'messages\', []) if m.get(\'role\') == \'user\']\n    return _content_text(users[-1].get(\'content\')) if users else \'\'\n\n\ndef rag_messages(messages, evidence):\n    """근거 조각을 담은 시스템 메시지를 맨 앞에 더한다. 서비스가 보낸 메시지는 바꾸지 않는다 (SPEC §2.6)."""\n    block = \'\\n\\n\'.join(f\'[{i + 1}] {e.text}\' for i, e in enumerate(evidence))\n    return [{\'role\': \'system\', \'content\': f\'{RAG_INSTRUCTION}\\n\\n<documents>\\n{block}\\n</documents>\'}] + list(messages)\n\n\ndef rag_reference(evidence):\n    """판정 근거 = LLM에 넣은 조각과 같은 텍스트."""\n    return \'\\n\\n\'.join(e.text for e in evidence)\n\n\ndef extract(body, default_mode=\'pass\', default_action=\'flag\', default_stride=0, with_stride=False, with_opts=False):\n    """반환: (upstream으로 보낼 body, reference 문자열 또는 None, mode, action[, stride][, opts])\n    with_stride=True면 앞부분 재판정 간격(stride)도 함께 돌려준다.\n    with_opts=True면 원래 verification 옵션도 돌려준다(collection 등). collection이 있고 근거가 없으면 mode를 off로 바꾸지 않는다."""\n    body = copy.deepcopy(body)\n    opts = body.pop(\'verification\', None) or {}\n\n    ref = opts.get(\'context\')\n    if isinstance(ref, list):\n        ref = \'\\n\\n\'.join(str(x) for x in ref)\n    if not ref:\n        found = []\n        for m in body.get(\'messages\', []):\n            found += CTX_TAG.findall(_content_text(m.get(\'content\')))\n        ref = \'\\n\\n\'.join(s.strip() for s in found) or None\n\n    mode = opts.get(\'mode\', default_mode)\n    action = opts.get(\'action\', default_action)\n    if mode not in MODES:\n        raise ValueError(f\'verification.mode must be one of {sorted(MODES)}\')\n    if action not in ACTIONS:\n        raise ValueError(f\'verification.action must be one of {sorted(ACTIONS)}\')\n    stride = opts.get(\'stride\', default_stride)\n    if not isinstance(stride, int) or isinstance(stride, bool) or stride < 0:\n        raise ValueError(\'verification.stride must be a non-negative integer\')\n    coll = opts.get(\'collection\')\n    if coll is not None and (not isinstance(coll, str) or not coll):\n        raise ValueError(\'verification.collection must be a non-empty string\')\n    if not ref and not coll:\n        mode = \'off\'\n    out = (body, ref, mode, action) + ((stride,) if with_stride else ()) + ((opts,) if with_opts else ())\n    return out\n'}
import sys
os.makedirs('/content/hg/hallugate', exist_ok=True)
open('/content/hg/hallugate/__init__.py', 'w').write('')
for n, s in SRC.items(): open(f'/content/hg/hallugate/{n}.py', 'w').write(s)
sys.path.insert(0, '/content/hg')
from hallugate.judge import StudentJudge, read_config, input_mode
def save_json(o, p): json.dump(o, open(p + '.tmp', 'w'), ensure_ascii=False); os.replace(p + '.tmp', p)
def load_json(p): return json.load(open(p)) if os.path.exists(p) else None
COMBO = COMBO_OLD
M1A = [('devL-haiku-13600', 11, 'contradiction', 'real'), ('devL-sonnet-15449', 7, 'outside_fact', 'ambiguous'), ('devL-haiku-14595', 10, 'contradiction', 'ambiguous'), ('devL-haiku-14106', 11, 'contradiction', 'real'), ('devL-haiku-14836', 10, 'other', 'ambiguous'), ('devL-haiku-14251', 11, 'contradiction', 'real'), ('devL-haiku-14071', 9, 'generalize', 'ambiguous'), ('devL-sonnet-14214', 2, 'numeric', 'ambiguous'), ('devL-haiku-14173', 9, 'contradiction', 'real'), ('devL-haiku-11942', 9, 'contradiction', 'real'), ('devL-haiku-13898', 1, 'other', 'real'), ('devL-haiku-14407', 11, 'outside_fact', 'ambiguous'), ('devL-haiku-14407', 13, 'contradiction', 'ambiguous'), ('devL-haiku-15598', 4, 'other', 'ambiguous'), ('devL-sonnet-12170', 9, 'contradiction', 'ambiguous'), ('devL-haiku-14554', 11, 'generalize', 'ambiguous'), ('devL-haiku-14971', 12, 'contradiction', 'real'), ('devL-sonnet-15104', 12, 'outside_fact', 'real'), ('devL-haiku-15569', 5, 'contradiction', 'real'), ('devL-haiku-14305', 10, 'outside_fact', 'ambiguous'), ('devL-haiku-14061', 4, 'other', 'ambiguous'), ('devL-haiku-14061', 5, 'contradiction', 'real'), ('devL-haiku-14768', 11, 'contradiction', 'real'), ('devL-sonnet-14606', 13, 'numeric', 'real'), ('devL-haiku-13890', 7, 'outside_fact', 'real'), ('devL-sonnet-14971', 13, 'contradiction', 'real'), ('devL-haiku-14514', 8, 'generalize', 'ambiguous'), ('devL-sonnet-11789', 1, 'outside_fact', 'real'), ('devL-haiku-13753', 8, 'contradiction', 'real'), ('devL-haiku-14232', 6, 'other', 'ambiguous'), ('devL-sonnet-13826', 3, 'other', 'ambiguous'), ('devL-haiku-14159', 10, 'contradiction', 'real'), ('devL-haiku-15697', 1, 'outside_fact', 'ambiguous'), ('devL-haiku-15697', 5, 'contradiction', 'real'), ('devL-haiku-14408', 3, 'outside_fact', 'ambiguous'), ('devL-sonnet-14863', 13, 'generalize', 'real'), ('devL-haiku-13622', 4, 'outside_fact', 'real'), ('devL-sonnet-15077', 13, 'numeric', 'real'), ('devL-haiku-12306', 3, 'contradiction', 'real'), ('devL-haiku-12306', 6, 'contradiction', 'real'), ('devL-haiku-12035', 4, 'contradiction', 'real'), ('devL-haiku-14784', 5, 'numeric', 'real'), ('devL-haiku-15726', 7, 'other', 'ambiguous'), ('devL-haiku-14636', 8, 'numeric', 'real'), ('devL-haiku-14493', 5, 'generalize', 'ambiguous'), ('devL-haiku-14493', 12, 'contradiction', 'real'), ('devL-haiku-12061', 3, 'other', 'ambiguous'), ('devL-haiku-12061', 4, 'outside_fact', 'real'), ('devL-sonnet-12037', 6, 'other', 'ambiguous'), ('devL-haiku-15736', 4, 'other', 'ambiguous'), ('devL-sonnet-14164', 11, 'contradiction', 'real'), ('devL-haiku-14883', 5, 'other', 'ambiguous'), ('devL-haiku-14883', 10, 'generalize', 'ambiguous'), ('devL-haiku-14979', 11, 'numeric', 'real'), ('devL-haiku-14979', 14, 'other', 'ambiguous'), ('devL-haiku-14999', 11, 'contradiction', 'real'), ('devL-haiku-14517', 11, 'contradiction', 'real'), ('devL-haiku-11836', 9, 'contradiction', 'real'), ('devL-haiku-11836', 10, 'generalize', 'ambiguous'), ('devL-haiku-14666', 7, 'contradiction', 'real'), ('devL-haiku-14645', 5, 'other', 'ambiguous'), ('devL-haiku-14645', 10, 'contradiction', 'real'), ('devL-sonnet-13723', 14, 'generalize', 'ambiguous'), ('devL-haiku-15077', 12, 'numeric', 'real'), ('devL-haiku-15183', 10, 'outside_fact', 'real'), ('devL-haiku-14944', 2, 'numeric', 'real'), ('devL-haiku-14944', 10, 'contradiction', 'real'), ('devL-haiku-14594', 6, 'numeric', 'real'), ('devL-haiku-14594', 10, 'contradiction', 'real'), ('devL-haiku-14370', 3, 'outside_fact', 'real'), ('devL-haiku-14370', 5, 'contradiction', 'real'), ('devL-haiku-14370', 7, 'outside_fact', 'real'), ('devL-haiku-14214', 7, 'contradiction', 'real'), ('devL-sonnet-11386', 6, 'generalize', 'ambiguous'), ('devL-haiku-14687', 9, 'contradiction', 'real'), ('devL-haiku-11789', 4, 'contradiction', 'real'), ('devL-haiku-12014', 2, 'contradiction', 'real'), ('devL-haiku-14534', 10, 'contradiction', 'real'), ('devL-haiku-15087', 11, 'contradiction', 'real'), ('devL-haiku-12089', 4, 'contradiction', 'real'), ('devL-haiku-12324', 21, 'outside_fact', 'real'), ('devL-haiku-15005', 9, 'contradiction', 'real'), ('devL-haiku-15005', 11, 'contradiction', 'real'), ('devL-haiku-13997', 11, 'numeric', 'real'), ('devL-sonnet-14294', 17, 'outside_fact', 'ambiguous'), ('devL-sonnet-15110', 19, 'outside_fact', 'ambiguous'), ('devL-haiku-12303', 8, 'outside_fact', 'ambiguous'), ('devL-haiku-13670', 10, 'contradiction', 'real'), ('devL-sonnet-15365', 11, 'outside_fact', 'real'), ('devL-haiku-14380', 15, 'contradiction', 'real'), ('devL-haiku-15065', 3, 'contradiction', 'real'), ('devL-haiku-14097', 5, 'outside_fact', 'real'), ('devL-haiku-14097', 6, 'contradiction', 'real'), ('devL-sonnet-14853', 8, 'generalize', 'ambiguous'), ('devL-sonnet-14251', 8, 'outside_fact', 'ambiguous'), ('devL-sonnet-14173', 11, 'outside_fact', 'ambiguous'), ('devL-haiku-14606', 9, 'contradiction', 'real'), ('devL-haiku-15365', 8, 'outside_fact', 'real'), ('devL-sonnet-14370', 7, 'other', 'ambiguous'), ('devL-haiku-13771', 12, 'contradiction', 'real'), ('devL-haiku-14522', 13, 'contradiction', 'real'), ('devL-haiku-14781', 6, 'contradiction', 'real'), ('devL-sonnet-15033', 8, 'outside_fact', 'ambiguous'), ('devL-haiku-12443', 2, 'numeric', 'real'), ('devL-haiku-15200', 16, 'outside_fact', 'real'), ('devL-sonnet-14666', 12, 'outside_fact', 'ambiguous'), ('devL-haiku-12052', 7, 'generalize', 'ambiguous')]

# 판정 입력: RAGTruth train 응답의 문장 끝 + 8토큰 지점 (게이트웨이와 같은 cl100k 토큰, 문장 시작부터)
import pysbd, tiktoken
ENC = tiktoken.get_encoding('cl100k_base')
SEG = pysbd.Segmenter(language='en', clean=False, char_span=True)

def build_reference(src):
    si, tt = src['source_info'], src['task_type']
    if tt == 'QA':
        return f"Question: {si.get('question','')}\n{si.get('passages','')}"
    if tt == 'Summary':
        return si if isinstance(si, str) else json.dumps(si, ensure_ascii=False)
    return '\n'.join(f"{k}: {json.dumps(v, ensure_ascii=False) if isinstance(v,(dict,list)) else v}"
                     for k, v in (si.items() if isinstance(si, dict) else []))

def points_of(text, s=8):
    # (문장 시작 글자, 지점 글자 끝) 목록: 문장마다 s토큰째 + 문장 끝
    ids = ENC.encode(text, disallowed_special=())
    _, starts = ENC.decode_with_offsets(ids)
    ends = starts[1:] + [len(text)]
    out = []
    for sp in SEG.segment(text):
        if not sp.sent.strip(): continue
        a = sp.start + (len(sp.sent) - len(sp.sent.lstrip()))
        tok = [e for st, e in zip(starts, ends) if a <= st < sp.end or (st < a < e)]
        for k, e in enumerate(tok, 1):
            if k % s == 0 or k == len(tok):
                out.append((a, min(e, sp.end)))
    return out

def make_inputs(responses, sources, n_resp=300, n_pts=300, seed=2027):
    rs = random.Random(seed)
    pool = [r for r in responses if r['split'] == 'train']
    cands = []
    for r in rs.sample(pool, min(n_resp, len(pool))):
        ref = build_reference(sources[r['source_id']])
        for a, cut in points_of(r['response']):
            text = r['response'][a:cut].strip()
            if text: cands.append({'ref': ref, 'text': text, 'prior': r['response'][:a], 'task': sources[r['source_id']]['task_type']})
    return rs.sample(cands, min(n_pts, len(cands)))

import contextlib, threading
from concurrent.futures import ThreadPoolExecutor

def make_judge(d, device):
    # load_judge와 같은 값으로 (판정기 폴더의 judge_config.json)
    cfg = read_config(d)
    return StudentJudge(d, embedder=cfg.get('embedder', 'sentence-transformers/all-MiniLM-L6-v2'), k=cfg.get('k', 8),
                        max_len=cfg.get('max_len', 512), chunk_words=cfg.get('chunk_words', 40), device=device,
                        mode=input_mode(cfg))

def combine(sf, si):
    # I3 조합: 두 점수를 cal-L 멀쩡한 응답 기준으로 표준화해 평균
    return ((sf - COMBO['F_mean']) / COMBO['F_std'] + (si - COMBO['I_mean']) / COMBO['I_std']) / 2

def judge_seq(jF, jI, refF, refI, x):
    # 차례로: 게이트웨이 judge() 두 번 (각각 토크나이즈·검색·순전파·동기화 포함)
    return jF.judge(refF, x['text'], x['prior']).score, jI.judge(refI, x['text'], x['prior']).score

STREAMS = {}
def _score_on_stream(j, ref, text, prior):
    # judge()와 같은 계산을 판정기 전용 CUDA 스트림에서. 동기화는 그 스트림만 (.cpu()가 현재 스트림을 기다림)
    ctx = torch.cuda.stream(STREAMS.setdefault(id(j), torch.cuda.Stream())) if j.device == 'cuda' else contextlib.nullcontext()
    with ctx, torch.no_grad():
        a, b = j._pair(ref, text, prior)
        enc = j.tok([a], [b], truncation='only_first', max_length=j.max_len, return_tensors='pt').to(j.device)
        with torch.autocast(j.device, enabled=(j.device == 'cuda')):
            return float(j.model(**enc).logits.squeeze().float().cpu())

POOL = ThreadPoolExecutor(2)
def judge_par(jF, jI, refF, refI, x):
    fF = POOL.submit(_score_on_stream, jF, refF, x['text'], x['prior'])
    fI = POOL.submit(_score_on_stream, jI, refI, x['text'], x['prior'])
    return fF.result(), fI.result()

def bench(fn, jF, jI, inputs, refs, warm=10):
    for x in inputs[:warm]: fn(jF, jI, *refs[x['ref']], x)
    ms, sc = [], []
    for x in inputs:
        t0 = time.perf_counter()
        sf, si = fn(jF, jI, *refs[x['ref']], x)
        c = combine(sf, si)
        ms.append((time.perf_counter() - t0) * 1000); sc.append((sf, si, c))
    return np.array(ms), np.array(sc)

def bench_one(j, inputs, refs, which, warm=10):
    for x in inputs[:warm]: j.judge(refs[x['ref']][which], x['text'], x['prior'])
    return np.array([j.judge(refs[x['ref']][which], x['text'], x['prior']).judge_ms for x in inputs])
SYSTEM = '''You are an expert annotator for RAGTruth, a benchmark of hallucinations in retrieval-augmented generation.
You receive a REFERENCE that a language model was given, and the model's RESPONSE split into numbered sentences.
Label EVERY sentence against the REFERENCE only (not world knowledge).

A sentence contains a hallucination if any part of it is:
- Evident conflict: clearly contradicts the reference (wrong fact, number, name, date, negation).
- Subtle conflict: deviates from the reference in a way that changes meaning (altered term, scope, time, attribution).
- Evident baseless info: introduces information that is not in the reference, even if it may be true in the real world.
- Subtle baseless info: adds unsupported inference, opinion, or embellishment (e.g. "popular", "cozy", "family-friendly") that the reference does not support.

Not hallucinations: faithful paraphrase; reasonable summarization; statements that the reference lacks the information or declining to answer;
meta phrases such as "Based on the passages,". For Data2txt (business data + user reviews), claims supported by the structured data or by what reviewers say are supported.

Labels: "S" = no hallucination. "H" = contains a hallucination. "A" = ambiguous, careful annotators could reasonably disagree.
For every H or A sentence also give: "type" (one of: evident_conflict, subtle_conflict, evident_baseless, subtle_baseless),
"phrase" (the shortest span copied EXACTLY, character for character, from that sentence that is hallucinated), and "reason" (at most 20 words).

Return ONLY a JSON object, no other text:
{"sentences": [{"i": 1, "label": "S"}, {"i": 2, "label": "H", "type": "evident_baseless", "phrase": "...", "reason": "..."}]}
with exactly one entry per numbered sentence, in order.'''

TASK_DESC = {'QA': 'QA — answer the question using the passages',
             'Summary': 'Summary — summarize the news article',
             'Data2txt': 'Data2txt — write an overview of a business from structured data and user reviews'}

def sentences(text):
    # (시작, 끝, 문장) — 판정 지점(points_of)과 같은 pysbd 분할
    out = []
    for sp in SEG.segment(text):
        if not sp.sent.strip(): continue
        a = sp.start + (len(sp.sent) - len(sp.sent.lstrip()))
        out.append((a, sp.start + len(sp.sent.rstrip()), sp.sent.strip()))
    return out

def label_msg(c, text):
    ss = sentences(text)
    lines = '\n'.join(f'[{i + 1}] {s}' for i, (_, _, s) in enumerate(ss))
    return f"TASK: {TASK_DESC[c['task']]}\n\nREFERENCE:\n<<<\n{c['ref_label']}\n>>>\n\nRESPONSE ({len(ss)} numbered sentences):\n{lines}"

def label_params(c, text):
    # R4와 같은 요청 (모델·max_tokens·지시문)
    return {'model': LABELER, 'max_tokens': 4096, 'system': SYSTEM, 'messages': [{'role': 'user', 'content': label_msg(c, text)}]}

def parse(text, n):
    try:
        d = json.loads(text[text.find('{'): text.rfind('}') + 1])
        labs = {}
        for e in d.get('sentences', []):
            i, lab = int(e.get('i', 0)), str(e.get('label', '')).strip().upper()[:1]
            if 1 <= i <= n and lab and lab in 'SHA':
                labs[i] = {'label': lab, 'type': e.get('type', ''), 'phrase': e.get('phrase', '') or '', 'reason': e.get('reason', '')}
        return labs, len(labs) == n
    except Exception:
        return {}, False

def score_response(jF, jI, refs, text, fn):
    # 판정 지점(문장마다 8토큰째 + 문장 끝)마다 (문장 시작, 지점 끝, F1 base, I2, 조합)
    out = []
    for a, cut in points_of(text):
        t = text[a:cut].strip()
        if not t: continue
        sf, si = fn(jF, jI, refs[0], refs[1], {'text': t, 'prior': text[:a]})
        out.append((a, cut, sf, si, combine(sf, si)))
    return out

def gen_status(g):
    if g is None: return 'nogen'
    if g['stop'] == 'refusal': return 'refusal'
    if not g['text'].strip() or not sentences(g['text']): return 'empty'
    return 'valid'

import pandas as pd

def wilson(k, n, z=1.96):
    if n == 0: return float('nan'), float('nan'), float('nan')
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h

def conformal(maxes, alpha=0.10):
    # cal-L 노트북(conformal_flag)과 같은 등각 보정: 멀쩡한 응답 최대 점수의 ceil((n+1)(1-α))번째 + 1e-6
    mx = np.sort(np.asarray(maxes, float)); n = len(mx); k = int(np.ceil((n + 1) * (1 - alpha)))
    return float(mx[k - 1]) + 1e-6 if k <= n else float('inf')

def response_rows(cases, gens, labels, scores, thr_c, thr_f):
    rows = []
    for cid, c in cases.items():
        g = gens.get(cid); st = gen_status(g)
        r = {'cid': cid, 'sid': c['sid'], 'task': c['task'], 'gen': c['gen'], 'flow': c['flow'], 'gold_in': c['gold_in'],
             'stop': g['stop'] if g else None, 'status': st}
        if st == 'valid':
            ss = sentences(g['text']); lab = labels.get(cid)
            r['status'] = 'ok' if lab is not None else 'nolabel'
            pts = scores.get(cid, [])
            r['n_sent'], r['n_pts'] = len(ss), len(pts)
            r['max_c'] = max([p[4] for p in pts], default=-np.inf)
            r['max_f'] = max([p[2] for p in pts], default=-np.inf)
            r['flag_c'], r['flag_f'] = bool(r['max_c'] >= thr_c), bool(r['max_f'] >= thr_f)
            first = next((p[0] for p in pts if p[4] >= thr_c), None)
            r['first_alarm_a'] = first
            if lab is not None:
                ls = [lab[i + 1]['label'] for i in range(len(ss))]
                r['n_H'], r['n_A'] = ls.count('H'), ls.count('A')
                r['clean'], r['strict_clean'] = r['n_H'] == 0, all(l == 'S' for l in ls)
                hs = [ss[i][0] for i, l in enumerate(ls) if l == 'H']
                r['first_H_a'] = hs[0] if hs else None
                r['strict_det'] = bool(hs and first is not None and first >= hs[0])
        rows.append(r)
    return pd.DataFrame(rows)

def rate_line(df, col='flag_c'):
    k, n = int(df[col].sum()), len(df)
    p, lo, hi = wilson(k, n)
    return {'k': k, 'n': n, 'rate': p, 'lo': lo, 'hi': hi}

def fmt(d):
    return f"{d['rate']:.3f} [{d['lo']:.3f}, {d['hi']:.3f}] ({d['k']}/{d['n']})" if d['n'] else '—'

def report(df, thr_name=''):
    ok = df[df.status == 'ok'].copy()
    for col in ['clean', 'strict_clean', 'flag_c', 'flag_f', 'strict_det']:
        ok[col] = ok[col].astype(bool)
    clean, hal = ok[ok.clean], ok[~ok.clean]
    out = {'status': df.status.value_counts().to_dict(), 'max_tokens_stops': int((df.stop == 'max_tokens').sum()),
           'clean_combo': rate_line(clean), 'clean_f1': rate_line(clean, 'flag_f'),
           'strict_clean_combo': rate_line(ok[ok.strict_clean]),
           'hal_any': rate_line(hal), 'hal_strict': rate_line(hal, 'strict_det'),
           'flag_precision_H': rate_line(ok[ok.flag_c], 'clean').copy(),
           'by_gen': {g: rate_line(d) for g, d in clean.groupby('gen')},
           'by_task': {t: rate_line(d) for t, d in clean.groupby('task')},
           'by_gen_task': {f'{g}|{t}': rate_line(d) for (g, t), d in clean.groupby(['gen', 'task'])},
           'hal_rate_by_gen': {g: float((~d.clean).mean()) for g, d in ok.groupby('gen')}}
    fp = out['flag_precision_H']                     # 표시된 응답 중 멀쩡한 비율 → H 비율로 바꿈
    if fp['n']: fp.update(k=fp['n'] - fp['k'], rate=1 - fp['rate'], lo=1 - fp['hi'], hi=1 - fp['lo'])
    qa = df[(df.flow == 'collection') & (df.gen == 'haiku')]
    out['qa_gold_in'] = float(qa.gold_in.astype(float).mean()) if len(qa) else None
    return out

# 최신 LLM 응답(M1 흐름) 다시 판정: 저장된 판정 지점·I2 점수는 그대로, F1 점수만 새 판정기(게이트웨이 judge.py)로
def zc(sf, si, C): return ((sf - C['F_mean']) / C['F_std'] + (si - C['I_mean']) / C['I_std']) / 2

def parse_labels(gens, raw):
    out = {}
    for c, r in raw.items():
        if c not in gens: continue
        labs, ok = parse(r['text'], len(sentences(gens[c]['text'])))
        if ok: out[c] = labs
    return out

def parity(j, cases, gens, old, n=30):
    # 기존 판정기로 저장 점수가 재현되는지 (지점 위치·앞 문맥이 M1과 같은지 확인)
    diffs = []
    for c in list(old)[:n]:
        g, ref = gens[c]['text'], j.prepare(cases[c]['ref_judge'])
        for a, cut, sf, si, _ in old[c][:3]:
            diffs.append(abs(j.judge(ref, g[a:cut].strip(), g[:a]).score - sf))
    return float(np.max(diffs)) if diffs else 0.0

def rescore(j, cases, gens, old, path, every=50):
    new = load_json(path) or {}
    todo = [c for c in old if c not in new]
    cache, t0 = {}, time.time()
    for k, c in enumerate(todo):
        r = cases[c]['ref_judge']
        if r not in cache: cache = {r: j.prepare(r)}
        g = gens[c]['text']
        new[c] = [[a, cut, j.judge(cache[r], g[a:cut].strip(), g[:a]).score, si, None] for a, cut, _, si, _ in old[c]]
        if (k + 1) % every == 0:
            save_json(new, path); print(f'  다시 판정 {len(new)}/{len(old)} ({time.time() - t0:.0f}s)', flush=True)
    save_json(new, path)
    return new

def with_combo(scores, C):
    return {c: [[a, cut, sf, si, zc(sf, si, C)] for a, cut, sf, si, _ in v] for c, v in scores.items()}

def paired(old_rows, new_rows, n_boot=2000, seed=2027):
    o = old_rows[old_rows.status == 'ok'].set_index('cid')
    n = new_rows.set_index('cid').loc[o.index]
    hal = ~o.clean.astype(bool)
    do, dn = o.flag_c[hal].astype(float).values, n.flag_c[hal].astype(float).values
    rng = np.random.default_rng(seed); m = len(do)
    bt = [dn[i].mean() - do[i].mean() for i in (rng.integers(0, m, m) for _ in range(n_boot))] if m else [np.nan]
    return {'n_hal': int(m), 'old': float(do.mean()) if m else np.nan, 'new': float(dn.mean()) if m else np.nan,
            'delta': float(dn.mean() - do.mean()) if m else np.nan,
            'ci_lo': float(np.percentile(bt, 2.5)), 'ci_hi': float(np.percentile(bt, 97.5))}

def sentence_flags(gens, scores, items, thr):
    # M1a 모순 문장: 그 문장의 판정 지점에서 경보가 울렸나
    out = []
    for cid, i, typ, val in items:
        if cid not in scores: continue
        a0 = sentences(gens[cid]['text'])[i - 1][0]
        out.append({'cid': cid, 'i': i, 'type': typ, 'validity': val,
                    'flag': bool(any(p[0] == a0 and p[4] >= thr for p in scores[cid]))})
    return pd.DataFrame(out)

def rows_of(R, model, kind):
    # model: 'old' | 'new', kind: 'A' | 'B'
    sc = R['old'] if model == 'old' else R['new']
    return response_rows(R['cases'], R['gens'], R['labels'], sc, THR[model][kind], THR_F[model])

def clean_max(R, model):
    d = response_rows(R['cases'], R['gens'], R['labels'], R['old'] if model == 'old' else R['new'], 0.0, 0.0)
    d = d[d.status == 'ok']
    return d[d.clean.astype(bool)].max_c.values

def k2_decide(RES):
    P = paired(rows_of(RES['devM'], 'old', 'B'), rows_of(RES['devM'], 'new', 'B'))
    rep_new_B = report(rows_of(RES['devM'], 'new', 'B'))
    k2_1 = bool(P['delta'] >= 0.10 and P['ci_lo'] > 0)
    k2_2 = bool(rep_new_B['clean_combo']['rate'] <= 0.15)
    return P, rep_new_B, k2_1, k2_2

THR_F = {'old': read_config(os.path.join(OLD_DIR, 'judge_base'))['thresholds']['8']['svc10_orig'], 'new': THR_KF}
jO = make_judge(os.path.join(OLD_DIR, 'judge_base'), DEVICE)
jK = make_judge(os.path.join(LOCAL, 'judge_base'), DEVICE)
assert jK.mode == 'retrieval'
RES = {}
for tag, mdir, suf in MSETS:
    cases, gens, raw, old = [load_json(os.path.join(mdir, f'{n}_{suf}.json')) for n in ['cases', 'gen', 'label', 'scores']]
    labels = parse_labels(gens, raw)
    pd_ = parity(jO, cases, gens, old)
    print(f'[{tag}] 응답 {len(old)} | 정답 파싱 {len(labels)}/{len(raw)} | 기존 판정기 재현 최대 차이 {pd_:.4f}')
    assert pd_ < 0.1, '저장 점수 재현 실패 — 판정 지점·앞 문맥이 저장 때와 다름'
    new = with_combo(rescore(jK, cases, gens, old, os.path.join(K2_DIR, f'scores_k2_{tag}.json')), COMBO_NEW)
    RES[tag] = {'cases': cases, 'gens': gens, 'labels': labels, 'old': with_combo(old, COMBO_OLD), 'new': new}
del jO; gc.collect()
THR = {'old': {'A': THR_OLD, 'B': conformal(clean_max(RES['calM'], 'old'))},
       'new': {'A': THR_NEW, 'B': conformal(clean_max(RES['calM'], 'new'))}}
print('임계값 (A = cal-L, B = cal-M 멀쩡한 응답 10%):', {m: {k: round(v, 4) for k, v in t.items()} for m, t in THR.items()})

## 8. 사전 기준 판정

In [ ]:
P, REP_NEW_B, K2_1, K2_2 = k2_decide(RES)
K2_0 = bool(K2_STATS['parse_ratio'] >= 0.90 and K2_STATS['H_sent'] >= 500 and len(clean_max(RES['calM'], 'old')) >= 400 and P['n_hal'] >= 150)
ADOPT = bool(K2_0 and K2_1 and K2_2 and K2_3)
DEPLOY = 'new' if ADOPT else 'old'
REPS = {(tag, m, k): report(rows_of(RES[tag], m, k)) for tag in RES for m in ['old', 'new'] for k in ['A', 'B']}
THR_B_OK = bool(REPS[('devM', DEPLOY, 'B')]['clean_combo']['rate'] <= 0.15)
for tag in ['devM', 'fresh', 'm1', 'calM']:
    print(f'[{tag}]')
    for m in ['old', 'new']:
        for k in ['A', 'B']:
            r = REPS[(tag, m, k)]
            print(f"  {m:3} {k} (임계값 {THR[m][k]:.3f}) | 환각 응답 탐지 {fmt(r['hal_any'])} | 첫 H 이후 {fmt(r['hal_strict'])} | 멀쩡한 응답 표시율 {fmt(r['clean_combo'])}")
    for t in TASKS:
        ro, rn = rows_of(RES[tag], 'old', 'B'), rows_of(RES[tag], 'new', 'B')
        h = (ro.status == 'ok') & (ro.task == t) & ~ro.clean.astype(bool)
        if h.sum():
            print(f"    {t:9} 환각 {int(h.sum()):3d} | 탐지(B) 기존 {ro.flag_c[h].astype(bool).mean():.3f} → 새 {rn.set_index('cid').loc[ro.cid[h]].flag_c.astype(bool).mean():.3f}")
SENT = {}
for m in ['old', 'new']:
    for k in ['A', 'B']:
        SENT[(m, k)] = sentence_flags(RES['m1']['gens'], RES['m1'][m], M1A, THR[m][k])
ct = {mk: d[d.type == 'contradiction'] for mk, d in SENT.items()}
print('(보고) M1a 직접 모순 문장 경보:', {f'{m}{k}': f"{int(d.flag.sum())}/{len(d)}" for (m, k), d in ct.items()})
print(f"\n=== K2-0: train 파싱 {K2_STATS['parse_ratio']:.3f}, H 문장 {K2_STATS['H_sent']}, cal-M 멀쩡 {len(clean_max(RES['calM'], 'old'))}, dev-M 환각 {P['n_hal']} → {'충족' if K2_0 else '미달 → 보류'} ===")
print(f"=== K2-1 (주): dev-M 환각 응답 탐지율(B) 기존 {P['old']:.3f} → 새 {P['new']:.3f} (Δ {P['delta']:+.3f} [{P['ci_lo']:+.3f}, {P['ci_hi']:+.3f}]; 기준 ≥ +0.10, 하한 > 0) → {'충족' if K2_1 else '미달'} ===")
print(f"=== K2-2 (안전, 최신 LLM): 새 조합(B) dev-M 멀쩡한 응답 표시율 {REP_NEW_B['clean_combo']['rate']:.3f} (≤ 0.15) → {'충족' if K2_2 else '미달'} ===")
print(f"=== K2-3 (안전, RAGTruth): {'충족' if K2_3 else '미달'} ===")
print(f"=== K2: {'채택 → 배포 판정기 = 새 조합' if ADOPT else '미채택 → 배포 판정기 = 기존 조합'} | 임계값 B({DEPLOY}): dev-M 멀쩡한 응답 표시율 "
      f"{REPS[('devM', DEPLOY, 'B')]['clean_combo']['rate']:.3f} → {'B 채택' if THR_B_OK else 'B 미채택(A 유지)'} ===")

## 9. 저장과 zip 다운로드

In [ ]:
OUTL = '/content/k2_out'; os.makedirs(OUTL, exist_ok=True)
cal_tab.to_csv(os.path.join(OUTL, 'k2_calL.csv'), index=False)
dev_tab.to_csv(os.path.join(OUTL, 'k2_devL.csv'), index=False)
for tag in RES:
    base = rows_of(RES[tag], 'old', 'A')
    for m in ['old', 'new']:
        for k in ['A', 'B']:
            if (m, k) == ('old', 'A'): continue
            r = rows_of(RES[tag], m, k)[['cid', 'max_c', 'max_f', 'flag_c', 'flag_f', 'first_alarm_a', 'strict_det']]
            base = base.merge(r.add_suffix(f'_{m}{k}').rename(columns={f'cid_{m}{k}': 'cid'}), on='cid', how='left')
    base.to_csv(os.path.join(OUTL, f'k2_{tag}.csv'), index=False)
pd.concat([d.assign(model=m, thr=k) for (m, k), d in SENT.items()]).to_csv(os.path.join(OUTL, 'k2_m1a_sentences.csv'), index=False)
summ = {'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu', 'train_k2': K2_STATS,
        'combo_old': COMBO_OLD, 'combo_new': COMBO_NEW, 'thr': THR, 'thr_k2_alone': THR_KF,
        'K2_0': K2_0, 'K2_1': K2_1, 'K2_2': K2_2, 'K2_3': K2_3, 'adopt': ADOPT, 'deploy': DEPLOY, 'thr_B_adopt': THR_B_OK,
        'devM_paired_B': P, 'calL_all_orig': _row('orig'), 'calL_all_C1': _row('C1'),
        'reports': {f'{t}|{m}|{k}': v for (t, m, k), v in REPS.items()},
        'm1a_contradiction': {f'{m}{k}': {'flag': int(d.flag.sum()), 'n': len(d)} for (m, k), d in ct.items()}}
json.dump(summ, open(os.path.join(OUTL, 'k2_summary.json'), 'w'), ensure_ascii=False, indent=1, default=float)
for f in os.listdir(OUTL):
    try: shutil.copy(os.path.join(OUTL, f), os.path.join(K2_DIR, f))
    except OSError as e: print('드라이브 저장 실패', f, e)
zp = shutil.make_archive('/content/k2_results', 'zip', OUTL)
from google.colab import files
files.download(zp)
print(json.dumps({k: summ[k] for k in ['K2_0', 'K2_1', 'K2_2', 'K2_3', 'adopt', 'thr_B_adopt']}, ensure_ascii=False))